## Phase 7 — Prior Case Retrieval: P7-0 Data inspection
Load Train/Dev/Test queries and candidate pools; report pool sizes, gold-set sizes,
document lengths and overlaps — these decide chunking, negatives and run cost.

In [5]:
# ---- P7-0: load the Phase 7 data and summarise it ----
import glob, json, numpy as np

# Locate Phase_7_PCR under the Kaggle inputs
root = glob.glob("/kaggle/input/**/Phase_7_PCR", recursive=True)
assert root, "Phase_7_PCR not found under /kaggle/input - add the dataset to the notebook"
R = root[0]; print("data root:", R)

def load(split):
    # Queries and candidate pool for one split
    q = json.load(open(f"{R}/{split}/{split.lower()}.json"))
    c = json.load(open(f"{R}/{split}/{split.lower()}_candidates.json"))
    return q, c

# Show the raw structure once so the loader matches it exactly
q, c = load("Train")
for name, obj in (("train.json", q), ("train_candidates.json", c)):
    first = obj[0] if isinstance(obj, list) else obj[next(iter(obj))]
    print(f"\n{name}: {type(obj).__name__} of {len(obj)}")
    print("  first record keys:", list(first.keys()) if isinstance(first, dict) else type(first).__name__)
    print("  sample:", json.dumps(first, ensure_ascii=False)[:400], "...")

def as_dict(obj, key="id"):
    # Normalise list-of-records or id->record into {id: record}
    return obj if isinstance(obj, dict) else {r[key]: r for r in obj}

def words(rec):
    # Document length in words (text = list of sentences)
    t = rec["text"] if isinstance(rec, dict) else rec
    return sum(len(s.split()) for s in t) if isinstance(t, list) else len(str(t).split())

# Per-split statistics
for split in ("Train", "Dev", "Test"):
    q, c = load(split); q, c = as_dict(q), as_dict(c)
    qlen = [words(r) for r in q.values()]; clen = [words(r) for r in c.values()]
    print(f"\n=== {split}: {len(q)} queries | pool {len(c)} candidates ===")
    print(f"  query words  median {int(np.median(qlen)):,}  p90 {int(np.percentile(qlen, 90)):,}  max {max(qlen):,}")
    print(f"  cand. words  median {int(np.median(clen)):,}  p90 {int(np.percentile(clen, 90)):,}  max {max(clen):,}")
    rel_key = next((k for k in next(iter(q.values())).keys() if "relevant" in k), None)
    if rel_key:
        g = [len(r[rel_key]) for r in q.values()]
        in_pool = np.mean([all(x in c for x in r[rel_key]) for r in q.values()])
        print(f"  gold per query  mean {np.mean(g):.2f}  median {np.median(g):.0f}  min {min(g)}  max {max(g)}"
              f"  | % queries with gold > 7: {100*np.mean(np.array(g) > 7):.1f}%  | all gold in pool: {100*in_pool:.0f}%")
        # Best possible Micro-F1@7 if the top 7 were perfect (caps from gold-set sizes)
        tp = sum(min(x, 7) for x in g); print(f"  oracle Micro-F1@7: {2*tp/(7*len(g) + sum(g)):.3f}")
    # Are query cases also inside their own pool? (must be excluded from rankings if so)
    print(f"  queries that also appear in the pool: {sum(k in c for k in q)}")

AssertionError: Phase_7_PCR not found under /kaggle/input - add the dataset to the notebook

In [6]:
# ---- P7-0a: find the Phase 7 data, or how earlier phase data was downloaded ----
import glob, os, subprocess
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()

# 1) Any Phase 7 folder, json or zip already present (inputs, working dir, home)
pats = ["Phase_7*", "*PCR*", "*pcr*", "*phase7*", "*Phase7*"]
found = sorted({p for base in ("/kaggle/input", "/kaggle/working", os.path.expanduser("~"))
                for pat in pats for p in glob.glob(f"{base}/**/{pat}", recursive=True)})
print("Phase 7 matches:", *found, sep="\n  ") if found else print("no Phase 7 files anywhere yet")

# 2) Which datasets are attached to this notebook at all
print("\n/kaggle/input contents:", sh("ls /kaggle/input") or "(nothing attached)")

# 3) How the Phase 6 data got here: search code and logs for its download source
print("\n--- how Phase_6_Summ was fetched ---")
print(sh("grep -rn -E 'Phase_6_Summ|gdown|drive.google|wget|curl|hf_hub_download|snapshot_download|unzip|\\.zip' "
         "/kaggle/working/src /kaggle/working/*.py 2>/dev/null | grep -v -i 'submission' | head -20")
      or "no download code in src/ - it was probably done in a notebook cell")

no Phase 7 files anywhere yet

/kaggle/input contents: (nothing attached)

--- how Phase_6_Summ was fetched ---
/kaggle/working/src/infer.py:104:    # strict checks, then prediction.json at the ROOT of <name>.zip
/kaggle/working/src/infer.py:113:    zp = f"{out_dir}/{name}.zip"
/kaggle/working/src/p6_run.py:101:    with zipfile.ZipFile(f"{folder}/p6_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:


In [7]:
# ---- P7-0b: is the Phase 7 data already in the HF backup repo? ----
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

REPO, REPO_TYPE = "Utkarsh-1718-1/cs781-phase1-work", "model"   # same as your restore cell
api = HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN"))

files = api.list_repo_files(REPO, repo_type=REPO_TYPE)
# Phase 7 files, and the raw Phase 6 data files for comparison (shows how earlier data was stored)
p7 = [f for f in files if any(k in f.lower() for k in ("phase_7", "pcr", "phase7"))]
p6 = [f for f in files if "phase_6" in f.lower() or "data_p6/phase" in f.lower()][:10]
print("Phase 7 files in repo:", *(p7 or ["none"]), sep="\n  ")
print("\nPhase 6 raw data in repo (for reference):", *(p6 or ["none"]), sep="\n  ")

Phase 7 files in repo:
  none

Phase 6 raw data in repo (for reference):
  none


## Phase 7 — P7-0a: Download the Phase 7 data
Fetch only the phase_7 files from the course dataset (Exploration-Lab/CS-781-Capstone) into data_p7/,
then back the raw files up to the HF work repo so a new session can restore them.

In [8]:
# ---- P7-0a: download the Phase 7 data from the course HF dataset ----
import os, subprocess
from huggingface_hub import HfApi, snapshot_download
from kaggle_secrets import UserSecretsClient

SRC_REPO = "Exploration-Lab/CS-781-Capstone"
DST = "/kaggle/working/data_p7"
tok = UserSecretsClient().get_secret("HF_WRITE_TOKEN")   # also works for reading gated datasets; never printed
api = HfApi(token=tok)

# 1) Which files in the course dataset belong to Phase 7 (folder names may be phase_7 or Phase_7_PCR)
files = api.list_repo_files(SRC_REPO, repo_type="dataset")
p7 = [f for f in files if "phase_7" in f.lower() or "pcr" in f.lower()]
print(f"{len(p7)} Phase 7 files:", *p7, sep="\n  ")
assert p7, "no Phase 7 files found - paste the 'Files and versions' folder names"

# 2) Download only those files (keeps the dataset's folder layout)
snapshot_download(SRC_REPO, repo_type="dataset", token=tok, local_dir=DST, allow_patterns=p7)

# 3) What landed on disk, with sizes
print("\n" + subprocess.run(f"cd {DST} && find . -type f -exec du -h {{}} + | sort -k2",
                            shell=True, capture_output=True, text=True).stdout)

# 4) Back the raw Phase 7 data up to the work repo (restored with everything else in future sessions)
api.upload_folder(folder_path=DST, path_in_repo="data_p7", repo_id="Utkarsh-1718-1/cs781-phase1-work",
                  repo_type="model", commit_message="Phase 7 raw data")
print("backed up to HF: data_p7/")

6 Phase 7 files:
  Phase_7_PCR/Dev/dev.json
  Phase_7_PCR/Dev/dev_candidates.json
  Phase_7_PCR/Test/test.json
  Phase_7_PCR/Test/test_candidates.json
  Phase_7_PCR/Train/train.json
  Phase_7_PCR/Train/train_candidates.json


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]


4.0K	./.cache/huggingface/CACHEDIR.TAG
4.0K	./.cache/huggingface/download/Phase_7_PCR/Dev/dev_candidates.json.metadata
4.0K	./.cache/huggingface/download/Phase_7_PCR/Dev/dev.json.metadata
4.0K	./.cache/huggingface/download/Phase_7_PCR/Test/test_candidates.json.metadata
4.0K	./.cache/huggingface/download/Phase_7_PCR/Test/test.json.metadata
4.0K	./.cache/huggingface/download/Phase_7_PCR/Train/train_candidates.json.metadata
4.0K	./.cache/huggingface/download/Phase_7_PCR/Train/train.json.metadata
4.0K	./.cache/huggingface/.gitignore
50M	./Phase_7_PCR/Dev/dev_candidates.json
5.6M	./Phase_7_PCR/Dev/dev.json
68M	./Phase_7_PCR/Test/test_candidates.json
9.0M	./Phase_7_PCR/Test/test.json
153M	./Phase_7_PCR/Train/train_candidates.json
32M	./Phase_7_PCR/Train/train.json



Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

backed up to HF: data_p7/


## Phase 7 — P7-0: Data inspection
Structure of queries and candidate pools; pool sizes, gold-set sizes, document lengths,
oracle Micro-F1@7, and whether queries / candidates overlap across splits.

In [9]:
# ---- P7-0: load the Phase 7 data and summarise it ----
import json, numpy as np

R = "/kaggle/working/data_p7/Phase_7_PCR"

def load(split):
    # Queries and candidate pool for one split
    s = split.lower()
    return json.load(open(f"{R}/{split}/{s}.json")), json.load(open(f"{R}/{split}/{s}_candidates.json"))

# Raw structure once, so the loader matches it exactly
q, c = load("Train")
for name, obj in (("train.json", q), ("train_candidates.json", c)):
    first = obj[0] if isinstance(obj, list) else obj[next(iter(obj))]
    print(f"{name}: {type(obj).__name__} of {len(obj)}")
    if isinstance(obj, dict): print("  first key:", next(iter(obj)))
    print("  record keys:", list(first.keys()) if isinstance(first, dict) else type(first).__name__)
    print("  sample:", json.dumps(first, ensure_ascii=False)[:350], "...\n")

def as_dict(obj):
    # Normalise list-of-records or id->record into {id: record}
    if isinstance(obj, dict): return obj
    idk = next(k for k in obj[0] if "id" in k.lower())
    return {r[idk]: r for r in obj}

def words(rec):
    # Document length in words (text = list of sentences)
    t = rec.get("text", rec) if isinstance(rec, dict) else rec
    return sum(len(s.split()) for s in t) if isinstance(t, list) else len(str(t).split())

D = {}
for split in ("Train", "Dev", "Test"):
    q, c = map(as_dict, load(split)); D[split] = (q, c)
    ql = [words(r) for r in q.values()]; cl = [words(r) for r in c.values()]
    print(f"=== {split}: {len(q)} queries | pool {len(c)} candidates ===")
    print(f"  query words  median {int(np.median(ql)):,}  p90 {int(np.percentile(ql, 90)):,}  max {max(ql):,}")
    print(f"  cand. words  median {int(np.median(cl)):,}  p90 {int(np.percentile(cl, 90)):,}  max {max(cl):,}")
    rk = next((k for k in next(iter(q.values())) if "relevant" in k), None)
    if rk:
        g = np.array([len(r[rk]) for r in q.values()])
        inpool = np.mean([all(x in c for x in r[rk]) for r in q.values()])
        tp = np.minimum(g, 7).sum()
        print(f"  gold/query mean {g.mean():.2f} median {np.median(g):.0f} min {g.min()} max {g.max()}"
              f" | >7 gold: {100*(g > 7).mean():.0f}% | all gold in pool: {100*inpool:.0f}%")
        print(f"  oracle Micro-F1@7 (perfect top-7): {2*tp/(7*len(g) + g.sum()):.3f}")
    print(f"  queries also inside their own pool: {sum(k in c for k in q)}\n")

# Overlap across splits (leakage check and whether pools are shared)
for a, b in (("Train", "Dev"), ("Train", "Test"), ("Dev", "Test")):
    print(f"{a}/{b}: shared queries {len(D[a][0].keys() & D[b][0].keys())} | "
          f"shared candidates {len(D[a][1].keys() & D[b][1].keys())}")

train.json: dict of 827
  first key: 0000000599
  record keys: ['text', 'relevant_candidates']
  sample: {"text": ["PETITIONER: <NAME> Vs. RESPONDENT: ROOP SINGH RATHORE & OTHERS(and connected appeal) DATE OF JUDGMENT: 07/05/1963 BENCH: <NAME>, S.K. BENCH: <NAME>, S.K. SINHA, BHUVNESHWAR P.(CJ) DAYAL, RAGHUBAR AYYANGAR, <NAME> MUDHOLKAR, J.R. CITATION: 1964 AIR 1545 1964 SCR (3) 573 CITATOR INFO : R 1970 SC 765 (9) R 1971 SC 342 (6) RF 1974 SC1185 (16 ...

train_candidates.json: dict of 4320
  first key: 0000000262
  record keys: ['text']
  sample: {"text": ["JUDGMENT <NAME>, C.J. ", "1. The only point which has to be considered in this appeal is whether or not the lower appellate Court was wrong in admitting in evidence statements contained in two Exhibits, being Ex. 58 and 59 in the suit. ", "2. The case of the plaintiffs in the suit was that they were the owners of the suit plot marked A B ...

=== Train: 827 queries | pool 4320 candidates ===
  query words  median 4,508  p90 12,602

## Phase 7 — P7-1: BM25 baseline, evaluator, submission writer
Lexical baseline (no model): BM25 over each split's own pool, self-match excluded.
Tuned on Train (no training involved), confirmed on Dev. Saves the BM25 top-100 per query
(hard negatives + re-ranking shortlist) and writes the first test submission.

In [10]:
# ---- P7-1: BM25 baseline + exact Micro-F1@7 evaluator + top-100 shortlists + submission ----
import json, os, time, zipfile, numpy as np
from sklearn.feature_extraction.text import CountVectorizer

R, OUT = "/kaggle/working/data_p7/Phase_7_PCR", "/kaggle/working/data_p7"
SUB = "/kaggle/working/submissions_p7"; os.makedirs(SUB, exist_ok=True)

def load(split):
    s = split.lower()
    return json.load(open(f"{R}/{split}/{s}.json")), json.load(open(f"{R}/{split}/{s}_candidates.json"))

def evaluate(rank, queries):
    """Exact competition metric: micro P/R/F1 over the top 7 of every query (gold neither truncated nor padded)."""
    tp = sum(len(set(rank[q][:7]) & set(r["relevant_candidates"])) for q, r in queries.items())
    n_gold = sum(len(r["relevant_candidates"]) for r in queries.values())
    P, Rc = tp / (7 * len(queries)), tp / n_gold
    return {"F1": 2 * P * Rc / (P + Rc + 1e-12), "P": P, "R": Rc}

def bm25_weights(tf, k1, b):
    """Document-side BM25 weights: idf * tf(k1+1) / (tf + k1(1 - b + b*dl/avgdl)), on a sparse doc x term matrix."""
    tf = tf.tocsr().astype(np.float32, copy=True)
    dl = np.asarray(tf.sum(1)).ravel(); N = tf.shape[0]
    df = np.bincount(tf.indices, minlength=tf.shape[1])
    idf = np.log1p((N - df + 0.5) / (df + 0.5)).astype(np.float32)
    rows = np.repeat(np.arange(N), np.diff(tf.indptr))
    v = tf.data
    tf.data = idf[tf.indices] * v * (k1 + 1) / (v + k1 * (1 - b + b * dl[rows] / dl.mean()))
    return tf

# Word tokens (letters only, length >= 2), English stop words removed; <NAME> placeholders drop out
vec_kw = dict(lowercase=True, token_pattern=r"(?u)\b[a-z][a-z]+\b", stop_words="english", dtype=np.float32)

cache = {}
def prepare(split):
    # Count matrices for one split: vocabulary from its own pool; queries mapped into it
    q, c = load(split)
    cids, qids = list(c), list(q)
    cv = CountVectorizer(**vec_kw)
    C = cv.fit_transform(" ".join(c[i]["text"]) for i in cids)
    Q = cv.transform(" ".join(q[i]["text"]) for i in qids)
    cache[split] = (q, qids, cids, C, Q)

def bm25_rank(split, k1=1.2, b=0.75, qmode="log", topn=100):
    q, qids, cids, C, Q = cache[split]
    W = bm25_weights(C, k1, b)
    Qw = Q.copy(); Qw.data = np.log1p(Qw.data) if qmode == "log" else np.ones_like(Qw.data)
    S = (Qw @ W.T).toarray()                                   # queries x candidates
    pos = {cid: j for j, cid in enumerate(cids)}
    for i, qid in enumerate(qids):                             # exclude the query itself from its own pool
        if qid in pos: S[i, pos[qid]] = -np.inf
    top = np.argsort(-S, axis=1)[:, :topn]
    return {qid: [cids[j] for j in top[i]] for i, qid in enumerate(qids)}, S

t = time.time()
for split in ("Train", "Dev", "Test"): prepare(split)
print(f"tokenised all splits in {time.time()-t:.0f}s")

# Is the query ever its own gold? (sanity check for the self-exclusion)
q, *_ = cache["Train"]
print("queries listing themselves as gold:", sum(k in r["relevant_candidates"] for k, r in q.items()))

# Tune on Train (no training happens, so its labels are a large, clean validation set)
res = []
for qmode in ("log", "binary"):
    for k1 in (0.9, 1.2, 1.6, 2.0):
        for b in (0.5, 0.75, 0.9):
            rank, _ = bm25_rank("Train", k1, b, qmode, topn=7)
            res.append((evaluate(rank, cache["Train"][0])["F1"], qmode, k1, b))
res.sort(reverse=True)
print("\nTrain grid (top 5):", *[f"F1 {f:.4f} | q={m} k1={k} b={bb}" for f, m, k, bb in res[:5]], sep="\n  ")
_, qmode, k1, b = res[0]

# Confirm on Dev with the chosen setting; also recall of the top-100 shortlist (ceiling for re-ranking)
for split in ("Train", "Dev"):
    rank, _ = bm25_rank(split, k1, b, qmode, topn=100)
    m = evaluate(rank, cache[split][0])
    qs = cache[split][0]
    rec = {n: np.mean([len(set(rank[k][:n]) & set(r["relevant_candidates"])) / max(1, len(r["relevant_candidates"]))
                       for k, r in qs.items() if r["relevant_candidates"]]) for n in (7, 20, 50, 100)}
    print(f"{split}: F1@7 {m['F1']:.4f}  P@7 {m['P']:.4f}  R@7 {m['R']:.4f} | "
          f"gold recall in top-n: " + "  ".join(f"@{n} {v:.2f}" for n, v in rec.items()))

# Save top-100 shortlists for every split (hard negatives + re-ranking input)
for split in ("Train", "Dev", "Test"):
    rank, _ = bm25_rank(split, k1, b, qmode, topn=100)
    json.dump(rank, open(f"{OUT}/bm25_top100_{split.lower()}.json", "w"))
json.dump({"k1": k1, "b": b, "qmode": qmode}, open(f"{OUT}/bm25_cfg.json", "w"))

def write_submission(rank, name, split="Test"):
    """prediction.json: exactly 7 unique in-pool ids per query, best first; zipped with the file at the root."""
    q, qids, cids, *_ = cache[split]; pool = set(cids)
    pred = {k: rank[k][:7] for k in qids}
    assert all(len(v) == 7 and len(set(v)) == 7 and set(v) <= pool and k not in v for k, v in pred.items())
    d = f"{SUB}/p7_{name}"; os.makedirs(d, exist_ok=True)
    json.dump(pred, open(f"{d}/prediction.json", "w"), indent=1)
    with zipfile.ZipFile(f"{d}/p7_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{d}/prediction.json", "prediction.json")
    print(f"submission: {d}/p7_{name}.zip ({len(pred)} queries)")

rank, _ = bm25_rank("Test", k1, b, qmode, topn=7)
write_submission(rank, "bm25")

tokenised all splits in 23s
queries listing themselves as gold: 0

Train grid (top 5):
  F1 0.1961 | q=log k1=2.0 b=0.9
  F1 0.1869 | q=log k1=1.6 b=0.9
  F1 0.1716 | q=log k1=1.2 b=0.9
  F1 0.1700 | q=log k1=2.0 b=0.75
  F1 0.1633 | q=binary k1=2.0 b=0.9
Train: F1@7 0.1961  P@7 0.1879  R@7 0.2050 | gold recall in top-n: @7 0.27  @20 0.40  @50 0.51  @100 0.59
Dev: F1@7 0.2401  P@7 0.2615  R@7 0.2220 | gold recall in top-n: @7 0.35  @20 0.47  @50 0.58  @100 0.67
submission: /kaggle/working/submissions_p7/p7_bm25/p7_bm25.zip (237 queries)


## Phase 7 — P7-1b: Finish the BM25 baseline; inspect the M1 / chunk-embedding code to reuse
Extend the BM25 grid past its edge (k1, b), refresh shortlists + submission if it improves;
print the tokenizer, M1 config and the Phase-4 chunk-embedding functions the retriever will reuse.

In [11]:
# ---- P7-1b: wider BM25 grid, then show the code the retriever will reuse ----
import subprocess, json
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout

# 1) BM25 grid beyond the previous edge (uses the cache and functions from P7-1)
res = []
for k1 in (2.0, 2.5, 3.0, 4.0):
    for b in (0.9, 0.95, 1.0):
        rank, _ = bm25_rank("Train", k1, b, "log", topn=7)
        res.append((evaluate(rank, cache["Train"][0])["F1"], k1, b))
res.sort(reverse=True)
print("Train grid:", *[f"F1 {f:.4f} | k1={k} b={bb}" for f, k, bb in res[:5]], sep="\n  ")
_, k1, b = res[0]
dev, _ = bm25_rank("Dev", k1, b, "log", topn=100)
print(f"Dev with k1={k1} b={b}: {evaluate(dev, cache['Dev'][0])}")

# Refresh the saved shortlists, config and test submission with the best setting
for split in ("Train", "Dev", "Test"):
    rank, _ = bm25_rank(split, k1, b, "log", topn=100)
    json.dump(rank, open(f"{OUT}/bm25_top100_{split.lower()}.json", "w"))
json.dump({"k1": k1, "b": b, "qmode": "log"}, open(f"{OUT}/bm25_cfg.json", "w"))
rank, _ = bm25_rank("Test", k1, b, "log", topn=7); write_submission(rank, "bm25")

# 2) Tokenizer: how earlier phases load the own BPE
print("\n--- tokenizer loading ---")
print(sh("grep -n -E '^(TK|tok|TOK)|Tokenizer|tokenizer.json|bpe' /kaggle/working/src/p6_sum.py /kaggle/working/src/p6_run.py | head -12"))

# 3) M1 checkpoint and its config (size decides batch / chunk budget)
print("--- M1 checkpoint + config ---")
print(sh("grep -n -E 'def load_lm|CK(PT)?\\s*=|ckpt.*m1|\\.pt' /kaggle/working/src/p6_run.py | head -8"))
print(sh("cd /kaggle/working/src && python3 - <<'EOF'\n"
         "import glob, torch\n"
         "for p in sorted(glob.glob('/kaggle/working/**/m1*.pt', recursive=True))[:6]:\n"
         "    ck = torch.load(p, map_location='cpu', weights_only=False)\n"
         "    if isinstance(ck, dict) and 'model_cfg' in ck:\n"
         "        print(p, '| arch', ck.get('arch'), '| cfg', ck['model_cfg'])\n"
         "EOF"))

# 4) Phase 4 chunk-embedding code (hidden states + pooling) and the model's forward signature
print("--- Phase 4 chunk embedding / hidden-state functions ---")
print(sh("grep -n -E '^def |^class |hidden|pool' /kaggle/working/src/p4_cjpe.py | head -40"))
print("--- model forward / hidden-state access (train.py) ---")
print(sh("grep -n -E 'def forward|def hidden|return_hidden|def encode|def features' /kaggle/working/src/train.py | head -12"))

Train grid:
  F1 0.2500 | k1=4.0 b=1.0
  F1 0.2379 | k1=4.0 b=0.95
  F1 0.2354 | k1=3.0 b=1.0
  F1 0.2278 | k1=4.0 b=0.9
  F1 0.2262 | k1=2.5 b=1.0
Dev with k1=4.0 b=1.0: {'F1': 0.3357420789322438, 'P': 0.36561743341404357, 'R': 0.3103802672147996}
submission: /kaggle/working/submissions_p7/p7_bm25/p7_bm25.zip (237 queries)

--- tokenizer loading ---
/kaggle/working/src/p6_run.py:35:from tokenizers import Tokenizer                                     # noqa: E402
/kaggle/working/src/p6_run.py:52:TK = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json")

--- M1 checkpoint + config ---
8:  {"name", "ckpt": "m1".."m4", "method": full|lora|lorma_plus|lorma_pi|houlsby|pfeiffer|prompt|qlora,
17:Outputs per run: stats/p6_runs/<n>.json (written LAST = done marker), p6_ckpt/<n>.pt (trained params),
40:CK = {"m1": f"{WORK}/checkpoints/m1_dense_100m/best.pt", "m2": f"{WORK}/checkpoints/m2_mla_moe/best.pt",
41:      "m3": f"{WORK}/checkpoints/m3_encdec/best.pt", "m4": f"{WORK}/checkpoint

## Phase 7 — P7-1c: BM25 k1 sweep; read the Phase-4 encoder code to reuse for retrieval

In [12]:
# ---- P7-1c: push BM25 k1 further, then print the Phase-4 encoder code the retriever builds on ----
import subprocess, json, torch
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout

# 1) k1 beyond 4 at b = 1.0 (b > 1 is not meaningful for length normalisation)
res = []
for k1 in (4.0, 5.0, 6.0, 8.0, 10.0):
    rank, _ = bm25_rank("Train", k1, 1.0, "log", topn=7)
    res.append((evaluate(rank, cache["Train"][0])["F1"], k1))
res.sort(reverse=True)
print("Train:", "  ".join(f"k1={k}: {f:.4f}" for f, k in sorted(res, key=lambda x: x[1])))
k1 = res[0][1]
dev, _ = bm25_rank("Dev", k1, 1.0, "log", topn=7)
print(f"best k1={k1} -> Dev {evaluate(dev, cache['Dev'][0])}")

# Save final BM25 shortlists / config / submission with the best k1
for split in ("Train", "Dev", "Test"):
    rank, _ = bm25_rank(split, k1, 1.0, "log", topn=100)
    json.dump(rank, open(f"{OUT}/bm25_top100_{split.lower()}.json", "w"))
json.dump({"k1": k1, "b": 1.0, "qmode": "log"}, open(f"{OUT}/bm25_cfg.json", "w"))
rank, _ = bm25_rank("Test", k1, 1.0, "log", topn=7); write_submission(rank, "bm25")

# 2) M1 config (size -> batch / chunk budget)
ck = torch.load("/kaggle/working/checkpoints/m1_dense_100m/best.pt", map_location="cpu", weights_only=False)
print("\nM1 arch:", ck.get("arch"), "| cfg:", ck.get("model_cfg")); del ck

# 3) Phase-4 encoder code to reuse: imports, Split, section, collate, DocClassifier, build, encode_chunks
for a, b_ in ((1, 37), (38, 133), (226, 248)):
    print(f"\n===== p4_cjpe.py lines {a}-{b_} =====")
    print(sh(f"sed -n '{a},{b_}p' /kaggle/working/src/p4_cjpe.py"))

Train: k1=4.0: 0.2500  k1=5.0: 0.2615  k1=6.0: 0.2691  k1=8.0: 0.2808  k1=10.0: 0.2860
best k1=10.0 -> Dev {'F1': 0.37687604224519544, 'P': 0.41041162227602906, 'R': 0.3484069886947585}
submission: /kaggle/working/submissions_p7/p7_bm25/p7_bm25.zip (237 queries)

M1 arch: dense | cfg: {}

===== p4_cjpe.py lines 1-37 =====
"""
p4_cjpe.py -- Phase 4 (Court Judgment Prediction, binary) on top of ANY Phase-1 backbone.

Data comes from the token cache written by P4-1 (data_p4/tok/{split}.npz: all tokens of every case,
uint16, plus offsets + labels; test labels are stored as -1 and never used).

Input views of a long case (our A1-BPE tokens):
  section(arr, "last",  k)  last k tokens           (the decision itself was removed from the texts,
  section(arr, "first", k)  first k tokens           so the reasoning at the end is expected to matter most)
  section(arr, "middle",k)  k tokens around the centre
  section(arr, "headtail",k) first k/4 + last 3k/4
DocClassifier: backbone (bidirectional)

## Phase 7 — P7-2: Retriever code (bi-encoder on M1 chunks) + runner
- Cases → 512-token chunks of our A1 BPE (up to 16 per case, evenly spread); chunk vector = M1 (bidirectional)
  masked mean pool; case vector = mean of normalised chunk vectors.
- Training: InfoNCE, τ = 0.05; per query 1 gold positive + 2 BM25 hard negatives + in-batch negatives;
  2 random chunks per case while training (memory); other gold cases in the batch are masked (no false negatives).
- Scoring over each split's FULL pool (self excluded): mean-pool cosine vs MaxSim (chunk late interaction),
  each alone and z-score-fused with BM25 (α tuned on Dev).
- Runner: background plans, skip-if-done, done-marker written last, per-run json/ckpt/embeddings/submission.

In [13]:
%%writefile /kaggle/working/src/p7_pcr.py
"""p7_pcr.py -- Phase 7 Prior Case Retrieval on the Phase-1 M1 backbone (bi-encoder over 512-token chunks).

Reuses p4_cjpe: build() (bidirectional M1 + adaptation method) and DocClassifier.embed() (masked mean pool).
Per run -> stats/p7_runs/<name>.json (written LAST = done marker), p7_ckpt/<name>.pt (trainable params),
p7_emb/<name>_{dev,test}.npz (chunk embeddings), submissions_p7/p7_<name>/p7_<name>.zip.
"""
import json, math, os, random, time, zipfile

import numpy as np
import torch
import torch.nn.functional as F

import p4_cjpe as cj

WORK = "/kaggle/working"
D7, R7 = f"{WORK}/data_p7", f"{WORK}/data_p7/Phase_7_PCR"
OUT, CKD, EMB, SUB = (f"{WORK}/stats/p7_runs", f"{WORK}/p7_ckpt", f"{WORK}/p7_emb", f"{WORK}/submissions_p7")
for d in (OUT, CKD, EMB, SUB):
    os.makedirs(d, exist_ok=True)
CKPT = f"{WORK}/checkpoints/m1_dense_100m/best.pt"
CHUNK = 512
# Learning rates per adaptation method (contrastive retrieval fine-tuning)
LR = {"full": 3e-5, "partial": 1e-4, "lora": 5e-4, "lorma_plus": 5e-4, "lorma_pi": 5e-4,
      "ia3": 3e-3, "prompt": 3e-3}


# ----------------------------------------------------------------------------- data
class Docs:
    """Token cache of every Phase-7 case (written by P7-2 cell 3): arr(case_id) -> uint16 BPE ids."""
    def __init__(self, path=f"{D7}/tok/docs.npz"):
        z = np.load(path)
        self.ids, self.off = z["ids"], z["offsets"]
        self.pos = {n: i for i, n in enumerate(z["names"].tolist())}

    def arr(self, d):
        i = self.pos[d]
        return self.ids[self.off[i]:self.off[i + 1]]


def load_split(split):
    """Queries {id: {...}} and the ordered candidate-id list of one split's own pool."""
    s = split.lower()
    q = json.load(open(f"{R7}/{split}/{s}.json"))
    c = json.load(open(f"{R7}/{split}/{s}_candidates.json"))
    return q, list(c)


def chunk_starts(n, max_chunks):
    """Start offsets of 512-token chunks; a tiny tail (< 64 tokens) is dropped; long cases keep
    `max_chunks` chunks spread evenly over the whole document (not just the beginning)."""
    st = list(range(0, max(1, n), CHUNK))
    if len(st) > 1 and n - st[-1] < 64:
        st = st[:-1]
    if len(st) > max_chunks:
        st = [st[i] for i in np.linspace(0, len(st) - 1, max_chunks).round().astype(int)]
    return st


# ----------------------------------------------------------------------------- model
def build(method, partial_k=4, r=16, alpha=32):
    """Bidirectional M1 + the adaptation method (Phase-4 builder; its 2-way head is unused here)."""
    kw = dict(r=r, alpha=alpha) if method in ("lora", "lorma_plus", "lorma_pi") else {}
    return cj.build(CKPT, method=method, bidirectional=True, partial_k=partial_k, **kw)


def embed_cases(model, docs, ids, k, device, train=True):
    """One unit vector per case = normalised mean of its normalised chunk vectors.
    train=True samples k random chunks (memory); train=False uses the first k evenly spread chunks."""
    seqs, owner = [], []
    for j, d in enumerate(ids):
        a = docs.arr(d)
        st = chunk_starts(len(a), 10 ** 9 if train else k)
        pick = random.sample(st, min(k, len(st))) if train else st
        for s in pick:
            seqs.append(a[s:s + CHUNK]); owner.append(j)
    x, mask = cj.collate(seqs, device)
    with cj._autocast(device):
        e = model.embed(x, mask)                                        # (chunks, d) fp32
    e = F.normalize(e, dim=-1)
    out = torch.zeros(len(ids), e.shape[1], device=device).index_add_(0, torch.tensor(owner, device=device), e)
    return F.normalize(out, dim=-1)


# ----------------------------------------------------------------------------- training
def train(model, method, epochs=3, bs=4, k=2, n_hard=2, tau=0.05, lr=None, device="cuda",
          log=print, seed=0, max_steps=None, teacher=None):
    """InfoNCE over [gold positive + BM25 hard negatives] of every query in the batch (in-batch negatives).
    Other gold cases of the same query and the query itself are masked out of the denominator."""
    random.seed(seed); torch.manual_seed(seed)
    q, _ = load_split("Train")
    bm = json.load(open(f"{D7}/bm25_top100_train.json"))
    items = [(qid, r["relevant_candidates"]) for qid, r in q.items() if r["relevant_candidates"]]
    docs = Docs()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr or LR[method], weight_decay=0.01)
    total = max_steps or epochs * math.ceil(len(items) / bs)
    warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    scaler = torch.amp.GradScaler("cuda", enabled=str(device).startswith("cuda"))
    model.train(); step, t0, hist, run_loss = 0, time.time(), [], 0.0
    for ep in range(max(1, epochs)):
        random.shuffle(items)
        for b in range(0, len(items), bs):
            batch = items[b:b + bs]
            qids, cand, tgt = [x[0] for x in batch], [], []
            for qid, gold in batch:
                g = set(gold)
                tgt.append(len(cand)); cand.append(random.choice(gold))
                hard = [c for c in bm[qid][:50] if c not in g and c != qid]
                cand += random.sample(hard, min(n_hard, len(hard)))
            qe = embed_cases(model, docs, qids, k, device)
            ce = embed_cases(model, docs, cand, k, device)
            S = qe @ ce.T / tau
            # Mask false negatives: other gold cases of query i, and query i itself, appearing as candidates
            bad = torch.zeros_like(S, dtype=torch.bool)
            for i, (qid, gold) in enumerate(batch):
                g = set(gold)
                for j, c in enumerate(cand):
                    if j != tgt[i] and (c in g or c == qid):
                        bad[i, j] = True
            loss = F.cross_entropy(S.masked_fill(bad, -1e4), torch.tensor(tgt, device=device))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            step += 1; run_loss += loss.item()
            if step % 50 == 0 or step == total:
                hist.append({"step": step, "loss": run_loss / min(50, step)}); run_loss = 0.0
                log(f"   step {step}/{total} | loss {hist[-1]['loss']:.4f} | {time.time()-t0:.0f}s")
            if step >= total:
                return hist
    return hist


# ----------------------------------------------------------------------------- encoding + scoring
@torch.no_grad()
def encode(model, docs, ids, max_chunks=16, device="cuda", bs=32):
    """Chunk embeddings of every case: E (n, max_chunks, d) fp16 unit vectors, M (n, max_chunks) mask."""
    model.eval()
    items = [(j, s) for j, d in enumerate(ids) for s in chunk_starts(len(docs.arr(d)), max_chunks)]
    d_model = model.tagger.backbone.cfg.d_model
    E = np.zeros((len(ids), max_chunks, d_model), np.float16); M = np.zeros((len(ids), max_chunks), bool)
    cnt = np.zeros(len(ids), int)
    for b in range(0, len(items), bs):
        batch = items[b:b + bs]
        x, mask = cj.collate([docs.arr(ids[j])[s:s + CHUNK] for j, s in batch], device)
        with cj._autocast(device):
            e = F.normalize(model.embed(x, mask), dim=-1).half().cpu().numpy()
        for (j, _), v in zip(batch, e):
            E[j, cnt[j]] = v; M[j, cnt[j]] = True; cnt[j] += 1
    return E, M


@torch.no_grad()
def dense_scores(E, M, qrows, mode, device="cuda"):
    """(n_queries, n_pool) similarities. mean: cosine of mean-pooled case vectors.
    maxsim: for each query chunk, best-matching candidate chunk; averaged over query chunks."""
    Et = torch.from_numpy(E).to(device).float(); Mt = torch.from_numpy(M).to(device)
    if mode == "mean":
        v = F.normalize((Et * Mt[..., None]).sum(1), dim=-1)
        return (v[qrows] @ v.T).cpu().numpy()
    out = np.zeros((len(qrows), len(E)), np.float32)
    flat = Et.reshape(-1, Et.shape[-1])                                 # all candidate chunks
    for i, r in enumerate(qrows):
        qc = Et[r][Mt[r]]                                               # (q_chunks, d)
        s = (qc @ flat.T).reshape(len(qc), len(E), -1)                  # (q_chunks, n_pool, max_chunks)
        s = s.masked_fill(~Mt[None], -1e4).max(-1).values.mean(0)
        out[i] = s.cpu().numpy()
    return out


def znorm(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-6)


def top7(S, qids, cids):
    """Rank each query's pool, excluding the query itself; best 7 ids."""
    pos = {c: j for j, c in enumerate(cids)}; S = S.copy()
    for i, qid in enumerate(qids):
        if qid in pos: S[i, pos[qid]] = -np.inf
    return {qid: [cids[j] for j in np.argsort(-S[i])[:7]] for i, qid in enumerate(qids)}


def evaluate(rank, queries):
    """Exact competition metric: micro P/R/F1 over the top 7 of every query."""
    tp = sum(len(set(rank[q][:7]) & set(r["relevant_candidates"])) for q, r in queries.items())
    ng = sum(len(r["relevant_candidates"]) for r in queries.values())
    P, Rc = tp / (7 * len(queries)), tp / ng
    return {"F1": 2 * P * Rc / (P + Rc + 1e-12), "P": P, "R": Rc}


def write_submission(rank, name, qids, cids):
    pool = set(cids); pred = {k: rank[k][:7] for k in qids}
    assert all(len(v) == 7 and len(set(v)) == 7 and set(v) <= pool and k not in v for k, v in pred.items())
    d = f"{SUB}/p7_{name}"; os.makedirs(d, exist_ok=True)
    json.dump(pred, open(f"{d}/prediction.json", "w"), indent=1)
    with zipfile.ZipFile(f"{d}/p7_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{d}/prediction.json", "prediction.json")


def bm25_matrix(split, qids, cids):
    """Saved full BM25 score matrix of the split, aligned to (qids, cids)."""
    z = np.load(f"{D7}/bm25_scores_{split.lower()}.npz")
    assert list(z["qids"]) == qids and list(z["cids"]) == cids, "BM25 matrix order mismatch"
    return z["S"]


def score_split(name, E, M, split, log=print, device="cuda", fixed=None):
    """Dense (mean / maxsim) alone and fused with BM25 over alpha. fixed=(mode, alpha) applies a Dev choice."""
    q, cids = load_split(split); qids = list(q); pos = {c: j for j, c in enumerate(cids)}
    qrows = [pos[x] for x in qids]
    Sb = znorm(bm25_matrix(split, qids, cids)); res = {}
    for mode in ([fixed[0]] if fixed else ("mean", "maxsim")):
        Sd = znorm(dense_scores(E, M, qrows, mode, device))
        for a in ([fixed[1]] if fixed else [round(x, 1) for x in np.arange(0, 1.01, 0.1)]):
            rank = top7(a * Sd + (1 - a) * Sb, qids, cids)
            res[(mode, a)] = (evaluate(rank, q) if split != "Test" else None, rank)
    return res, qids, cids


# ----------------------------------------------------------------------------- one experiment
def run(name, method="full", epochs=3, bs=4, k=2, n_hard=2, tau=0.05, lr=None, partial_k=4, r=16, alpha=32,
        max_chunks=16, seed=0, log=print, device="cuda"):
    t0 = time.time()
    model = build(method, partial_k, r, alpha).to(device)
    ntr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    ntot = sum(p.numel() for p in model.parameters())
    log(f"=== {name}: {method} | trainable {ntr/1e6:.3f}M of {ntot/1e6:.1f}M | epochs {epochs}")
    torch.cuda.reset_peak_memory_stats()
    hist = train(model, method, epochs, bs, k, n_hard, tau, lr, device, log, seed) if epochs > 0 else []
    train_min = (time.time() - t0) / 60
    if epochs > 0:
        torch.save({n: p.detach().half().cpu() for n, p in model.named_parameters() if p.requires_grad},
                   f"{CKD}/{name}.pt")
    docs, emb, enc_s = Docs(), {}, {}
    for split in ("Dev", "Test"):
        _, cids = load_split(split); t = time.time()
        emb[split] = encode(model, docs, cids, max_chunks, device); enc_s[split] = time.time() - t
        np.savez(f"{EMB}/{name}_{split.lower()}.npz", E=emb[split][0], M=emb[split][1])
        log(f"   encoded {split}: {len(cids)} cases in {enc_s[split]:.0f}s")
    # Dev: every (pooling, alpha); pick the best -> apply unchanged to Test
    dev, _, _ = score_split(name, *emb["Dev"], "Dev", log, device)
    table = {f"{m}|{a}": v[0] for (m, a), v in dev.items()}
    best = max(dev, key=lambda key: dev[key][0]["F1"])
    for m in ("mean", "maxsim"):
        log(f"   Dev {m:6s}: dense-only F1 {dev[(m, 1.0)][0]['F1']:.4f} | "
            f"best fused {max(dev[(m, a)][0]['F1'] for a in [round(x, 1) for x in np.arange(0, 1.01, 0.1)]):.4f}")
    test, qids, cids = score_split(name, *emb["Test"], "Test", log, device, fixed=best)
    write_submission(test[best][1], name, qids, cids)
    rec = {"name": name, "method": method,
           "cfg": dict(epochs=epochs, bs=bs, k=k, n_hard=n_hard, tau=tau, lr=lr or LR.get(method),
                       partial_k=partial_k, r=r, alpha=alpha, max_chunks=max_chunks, seed=seed),
           "trainable_M": round(ntr / 1e6, 3), "total_M": round(ntot / 1e6, 1),
           "peak_mem_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2),
           "train_min": round(train_min, 1), "encode_s": {s: round(v) for s, v in enc_s.items()},
           "dev_best": {"mode": best[0], "alpha": best[1], **dev[best][0]},
           "dev_dense_only": {m: dev[(m, 1.0)][0] for m in ("mean", "maxsim")},
           "dev_table": table, "hist": hist, "min": round((time.time() - t0) / 60, 1)}
    json.dump(rec, open(f"{OUT}/{name}.json", "w"), indent=1)          # written LAST = done marker
    log(f"DONE {name}: Dev F1 {dev[best][0]['F1']:.4f} (P {dev[best][0]['P']:.3f} R {dev[best][0]['R']:.3f}) "
        f"| {best[0]} alpha {best[1]} | {rec['min']} min")
    return rec


def smoke(log=print, device="cuda"):
    """Every method: build, 2 training steps, encode 6 cases -- catches wiring errors in ~2 minutes."""
    docs = Docs(); _, cids = load_split("Dev")
    for method in ("full", "partial", "lorma_plus", "ia3", "prompt"):
        m = build(method).to(device)
        ntr = sum(p.numel() for p in m.parameters() if p.requires_grad)
        hist = train(m, method, bs=2, k=1, n_hard=1, device=device, log=lambda *a: None, max_steps=2)
        E, M = encode(m, docs, cids[:6], max_chunks=2, device=device)
        log(f"smoke {method:10s} trainable {ntr/1e6:8.3f}M | 2 steps ok | emb {E.shape} | "
            f"norms {np.linalg.norm(E[M].astype(np.float32), axis=-1).mean():.3f}")
        del m; torch.cuda.empty_cache()
    log("SMOKE TEST PASSED")

Writing /kaggle/working/src/p7_pcr.py


In [14]:
%%writefile /kaggle/working/src/p7_run.py
"""p7_run.py -- run a Phase-7 plan in the background:  python3 p7_run.py --plan stats/p7_x.json  [--smoke]
Plan = list of {"name": ..., "method": ..., <run() kwargs>}; finished runs (stats/p7_runs/<name>.json) are skipped."""
import argparse, json, os, sys, time, traceback

import torch

sys.path.insert(0, "/kaggle/working/src")
import p7_pcr as P                                                      # noqa: E402

ap = argparse.ArgumentParser(); ap.add_argument("--plan"); ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


if args.smoke:
    P.smoke(log); sys.exit()
for job in json.load(open(args.plan)):
    name = job.pop("name")
    if os.path.exists(f"{P.OUT}/{name}.json"):
        log(f"skip {name} (already done)"); continue
    try:
        P.run(name, log=log, **job)
    except Exception:
        log(f"!!! {name} FAILED"); traceback.print_exc(); sys.stdout.flush()
    torch.cuda.empty_cache()
log("PLAN DONE")

Writing /kaggle/working/src/p7_run.py


In [15]:
# ---- P7-2c: final BM25 (k1 sweep) + full score matrices, tokenise all cases with our BPE, smoke test ----
import json, numpy as np, subprocess, time
from tokenizers import Tokenizer

D7 = "/kaggle/working/data_p7"; R7 = f"{D7}/Phase_7_PCR"

# 1) BM25: k1 beyond 10 at b = 1.0, choose on Train, report Dev
res = sorted(((evaluate(bm25_rank("Train", k, 1.0, "log", topn=7)[0], cache["Train"][0])["F1"], k)
              for k in (10.0, 15.0, 20.0, 30.0, 50.0)), reverse=True)
print("Train:", "  ".join(f"k1={k}: {f:.4f}" for f, k in sorted(res, key=lambda x: x[1])))
k1 = res[0][1]
print(f"final BM25 k1={k1} b=1.0 -> Dev {evaluate(bm25_rank('Dev', k1, 1.0, 'log', topn=7)[0], cache['Dev'][0])}")

# Save shortlists, full score matrices (for fusion), config and the BM25 submission with the final setting
for split in ("Train", "Dev", "Test"):
    rank, S = bm25_rank(split, k1, 1.0, "log", topn=100)
    json.dump(rank, open(f"{D7}/bm25_top100_{split.lower()}.json", "w"))
    q, qids, cids, *_ = cache[split]
    np.savez(f"{D7}/bm25_scores_{split.lower()}.npz", S=np.where(np.isinf(S), -1e9, S).astype(np.float32),
             qids=np.array(qids), cids=np.array(cids))
json.dump({"k1": k1, "b": 1.0, "qmode": "log"}, open(f"{D7}/bm25_cfg.json", "w"))
write_submission(bm25_rank("Test", k1, 1.0, "log", topn=7)[0], "bm25")

# 2) Tokenise every Phase-7 case once with the A1 BPE (candidate pools contain all queries too)
TK = Tokenizer.from_file("/kaggle/working/tokenizer/hf_a1_bpe_v2_32k.json")
texts = {}
for split in ("Train", "Dev", "Test"):
    for d, r in json.load(open(f"{R7}/{split}/{split.lower()}_candidates.json")).items():
        texts.setdefault(d, " ".join(r["text"]))
names = sorted(texts); arrs = []; t = time.time()
for i in range(0, len(names), 200):                                    # batches keep RAM in check
    for e in TK.encode_batch([texts[n] for n in names[i:i + 200]]):
        arrs.append(np.array(e.ids[:131072], dtype=np.uint16))         # cap at 128k tokens (256 chunks)
lens = np.array([len(a) for a in arrs])
import os; os.makedirs(f"{D7}/tok", exist_ok=True)
np.savez(f"{D7}/tok/docs.npz", ids=np.concatenate(arrs), offsets=np.concatenate([[0], np.cumsum(lens)]),
         names=np.array(names))
print(f"\ntokenised {len(names)} cases in {time.time()-t:.0f}s | tokens median {int(np.median(lens)):,} "
      f"p90 {int(np.percentile(lens, 90)):,} | total {lens.sum()/1e6:.1f}M | "
      f"cases > 16 chunks: {100*np.mean(lens > 16*512):.0f}%")

# 3) Smoke test on GPU 0: every method builds, trains 2 steps, encodes (catches wiring errors before real runs)
print(subprocess.run("cd /kaggle/working/src && CUDA_VISIBLE_DEVICES=0 python3 p7_run.py --smoke 2>&1 | tail -n 25",
                     shell=True, capture_output=True, text=True).stdout)

Train: k1=10.0: 0.2860  k1=15.0: 0.2923  k1=20.0: 0.2938  k1=30.0: 0.2923  k1=50.0: 0.2898
final BM25 k1=20.0 b=1.0 -> Dev {'F1': 0.3935519733180136, 'P': 0.42857142857142855, 'R': 0.36382322713257964}
submission: /kaggle/working/submissions_p7/p7_bm25/p7_bm25.zip (237 queries)

tokenised 5825 cases in 59s | tokens median 4,552 p90 15,096 | total 43.9M | cases > 16 chunks: 26%
20:14:07 smoke full       trainable  100.096M | 2 steps ok | emb (6, 2, 768) | norms 1.000
20:14:11 smoke partial    trainable   25.174M | 2 steps ok | emb (6, 2, 768) | norms 1.000
20:14:15 smoke lorma_plus trainable    0.395M | 2 steps ok | emb (6, 2, 768) | norms 1.000
20:14:19 smoke ia3        trainable    0.032M | 2 steps ok | emb (6, 2, 768) | norms 1.000
20:14:23 smoke prompt     trainable    0.017M | 2 steps ok | emb (6, 2, 768) | norms 1.000
20:14:23 SMOKE TEST PASSED



## Phase 7 — P7-3: Launch experiments (both GPUs, background, HF backup after every run)
GPU 0: full (also the KD teacher) → LoRMA+ → partial (top-4 layers)
GPU 1: zero-shot M1 (no training: Phase-1 transfer) → IA³ → prompt tuning

In [16]:
# ---- P7-3: write plans and launch the first Phase-7 experiments on both GPUs (background + HF backups) ----
import os, json, subprocess
from kaggle_secrets import UserSecretsClient

W, S = "/kaggle/working", "/kaggle/working/stats"
REPO, REPO_TYPE = "Utkarsh-1718-1/cs781-phase1-work", "model"

# HF backup script: code, results, data cache, embeddings, submissions + named checkpoints (retries on clashes)
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_WRITE_TOKEN")      # env only, never printed
open(f"{W}/hf_backup.py", "w").write(f'''
import sys, os, glob, time
from huggingface_hub import HfApi
api, W = HfApi(token=os.environ["HF_TOKEN"]), "{W}"
paths = [f"{{W}}/{{d}}" for d in ("src", "stats", "data_p7/tok", "submissions_p7", "p7_emb")]
paths += glob.glob(f"{{W}}/data_p7/bm25_*")
paths += [p for n in sys.argv[1:] for p in glob.glob(f"{{W}}/p7_ckpt/{{n}}*")]
for p in paths:
    if not os.path.exists(p): continue
    rel = os.path.relpath(p, W)
    for attempt in range(3):
        try:
            kw = dict(path_in_repo=rel, repo_id="{REPO}", repo_type="{REPO_TYPE}", commit_message="auto backup " + rel)
            api.upload_folder(folder_path=p, **kw) if os.path.isdir(p) else api.upload_file(path_or_fileobj=p, **kw)
            break
        except Exception as e:
            print("retry", rel, type(e).__name__); time.sleep(30)
print("backup ok", sys.argv[1:])
''')

# Plans: same recipe for every technique (3 epochs, 2 chunks/case in training, 16 chunks at encode time)
PLANS = {
    0: [{"name": "p7_full",       "method": "full",       "epochs": 3},
        {"name": "p7_lorma_plus", "method": "lorma_plus", "epochs": 3},
        {"name": "p7_partial",    "method": "partial",    "epochs": 3, "partial_k": 4}],
    1: [{"name": "p7_zeroshot",   "method": "full",       "epochs": 0},          # untrained Phase-1 M1
        {"name": "p7_ia3",        "method": "ia3",        "epochs": 3},
        {"name": "p7_prompt",     "method": "prompt",     "epochs": 3}],
}
for gpu, jobs in PLANS.items():
    steps = []
    for j in jobs:                                    # one plan file per run -> back up right after each
        pf = f"{S}/p7plan_{j['name']}.json"; json.dump([j], open(pf, "w"))
        steps += [f"python3 p7_run.py --plan {pf}", f"python3 {W}/hf_backup.py {j['name']}"]
    chain = f"cd {W}/src && export CUDA_VISIBLE_DEVICES={gpu}; " + "; ".join(steps)
    subprocess.run(f'nohup bash -c "{chain}" >> {S}/p7_gpu{gpu}.log 2>&1 &', shell=True)
    print(f"GPU {gpu}: {[j['name'] for j in jobs]} -> stats/p7_gpu{gpu}.log")

# First backup of the data cache + BM25 files right away (CPU, background)
subprocess.run(f"nohup python3 {W}/hf_backup.py > {S}/p7_backup0.log 2>&1 &", shell=True)

GPU 0: ['p7_full', 'p7_lorma_plus', 'p7_partial'] -> stats/p7_gpu0.log
GPU 1: ['p7_zeroshot', 'p7_ia3', 'p7_prompt'] -> stats/p7_gpu1.log


CompletedProcess(args='nohup python3 /kaggle/working/hf_backup.py > /kaggle/working/stats/p7_backup0.log 2>&1 &', returncode=0)

In [4]:
# ---- P7 monitor: GPU use, current job per GPU, latest log lines, finished runs ranked by Dev F1@7 ----
import subprocess, glob, json, os, time
W, S = "/kaggle/working", "/kaggle/working/stats"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()

print(sh("nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader"), "\n")
for f in sorted(glob.glob(f"{S}/p7_gpu*.log")):
    lines = [l for l in open(f, errors="ignore").read().splitlines() if l.strip() and "%|" not in l
             and "Upload" not in l and "Processing" not in l]
    age = (time.time() - os.path.getmtime(f)) / 60
    print(f"=== {os.path.basename(f)} | last write {age:.0f} min ago ===", *lines[-4:], sep="\n")

rows = []
for f in glob.glob(f"{S}/p7_runs/*.json"):
    r = json.load(open(f)); b = r["dev_best"]
    rows.append((b["F1"], r["name"], b["P"], b["R"], b["mode"], b["alpha"],
                 r["dev_dense_only"]["mean"]["F1"], r["dev_dense_only"]["maxsim"]["F1"], r["trainable_M"], r["min"]))
print(f"\n{'run':<16}{'F1@7':>7}{'P':>7}{'R':>7}  {'best':<7}{'α':>4} | {'dense mean':>10}{'maxsim':>8} | {'train M':>8}{'min':>6}")
print(f"{'bm25 (Dev)':<16}{0.3936:7.4f}{0.4286:7.3f}{0.3638:7.3f}")
for r in sorted(rows, reverse=True):
    print(f"{r[1]:<16}{r[0]:7.4f}{r[2]:7.3f}{r[3]:7.3f}  {r[4]:<7}{r[5]:4.1f} | {r[6]:10.4f}{r[7]:8.4f} | {r[8]:8.3f}{r[9]:6.1f}")

0, 0 %, 0 MiB
1, 99 %, 7261 MiB 

=== p7_gpu0.log | last write 6 min ago ===
    api, W = HfApi(token=os.environ["HF_TOKEN"]), "/kaggle/working"
                         ~~~~~~~~~~^^^^^^^^^^^^
  File "<frozen os>", line 714, in __getitem__
KeyError: 'HF_TOKEN'
=== p7_gpu1.log | last write 3 min ago ===
03:03:19    step 550/615 | loss 1.6524 | 576s
03:04:13    step 600/615 | loss 1.5541 | 629s
03:04:28    step 615/615 | loss 0.4512 | 644s
03:06:41    encoded Dev: 1023 cases in 133s

run                F1@7      P      R  best      α | dense mean  maxsim |  train M   min
bm25 (Dev)       0.3936  0.429  0.364
p7_lorma_plus    0.4091  0.446  0.378  maxsim  0.7 |     0.3413  0.3591 |    0.395  13.7
p7_kd_m1x6       0.4024  0.438  0.372  maxsim  0.4 |     0.3291  0.3424 |   62.337  10.7
p7_full          0.4024  0.438  0.372  maxsim  0.4 |     0.3513  0.3513 |  100.096  15.2
p7_partial       0.4013  0.437  0.371  mean    0.6 |     0.3491  0.3469 |   25.174  10.2
p7_ia3           0.3980  0.433

## Phase 7 — P7-3b: Fix fusion (self-column distorted the z-norm) and rescore all runs from saved embeddings
The query's own column (−1e9 in the BM25 matrix) entered the per-query z-score and flattened BM25,
so every fused score collapsed to "BM25 only". Fix: self column → NaN, nan-aware z-score; α grid in 0.05 steps.
Dense-only numbers are unchanged; fused numbers and test submissions are recomputed (no retraining).

In [27]:
# ---- P7-3b: patch the fusion in p7_pcr.py and rescore every finished run from its saved embeddings ----
import sys, json, glob, importlib, subprocess, os
W, S = "/kaggle/working", "/kaggle/working/stats"
P = f"{W}/src/p7_pcr.py"; src = open(P).read()

new_znorm = '''ALPHAS = [round(float(x), 2) for x in np.arange(0, 1.0001, 0.05)]


def znorm(S):
    """Per-query z-score over the pool, ignoring NaN (the query's own column)."""
    return (S - np.nanmean(S, 1, keepdims=True)) / (np.nanstd(S, 1, keepdims=True) + 1e-6)


'''
new_score = '''def score_split(name, E, M, split, log=print, device="cuda", fixed=None):
    """Dense (mean / maxsim) alone and fused with BM25 over alpha; the query's own column is NaN before
    z-scoring (so it cannot distort either distribution) and -inf at ranking. fixed=(mode, alpha) for Test."""
    q, cids = load_split(split); qids = list(q); pos = {c: j for j, c in enumerate(cids)}
    qrows = np.array([pos[x] for x in qids]); ar = np.arange(len(qids))
    Sb = bm25_matrix(split, qids, cids).astype(np.float32); Sb[ar, qrows] = np.nan; Sb = znorm(Sb)
    res = {}
    for mode in ([fixed[0]] if fixed else ("mean", "maxsim")):
        Sd = dense_scores(E, M, list(qrows), mode, device).astype(np.float32)
        Sd[ar, qrows] = np.nan; Sd = znorm(Sd)
        for a in ([fixed[1]] if fixed else ALPHAS):
            rank = top7(np.nan_to_num(a * Sd + (1 - a) * Sb, nan=-np.inf), qids, cids)
            res[(mode, a)] = (evaluate(rank, q) if split != "Test" else None, rank)
    return res, qids, cids


def rescore(name, log=print, device="cuda"):
    """Re-run Dev selection + Test submission for a finished run from its saved chunk embeddings."""
    rec = json.load(open(f"{OUT}/{name}.json"))
    emb = {}
    for s in ("Dev", "Test"):
        z = np.load(f"{EMB}/{name}_{s.lower()}.npz"); emb[s] = (z["E"], z["M"])
    dev, _, _ = score_split(name, *emb["Dev"], "Dev", log, device)
    best = max(dev, key=lambda k: dev[k][0]["F1"])
    test, qids, cids = score_split(name, *emb["Test"], "Test", log, device, fixed=best)
    write_submission(test[best][1], name, qids, cids)
    rec.update(dev_best={"mode": best[0], "alpha": best[1], **dev[best][0]},
               dev_dense_only={m: dev[(m, 1.0)][0] for m in ("mean", "maxsim")},
               dev_table={f"{m}|{a}": v[0] for (m, a), v in dev.items()},
               rescored="self column NaN in z-norm; alpha step 0.05")
    json.dump(rec, open(f"{OUT}/{name}.json", "w"), indent=1)
    return rec


'''
# Swap in the fixed znorm and score_split (+ rescore); make run() use the finer alpha grid too
a, b = src.index("def znorm(S):"), src.index("def top7(")
src = src[:a] + new_znorm + src[b:]
a = src.index("def score_split("); b = src.index("# ------", a)
src = src[:a] + new_score + src[b:]
src = src.replace("[round(x, 1) for x in np.arange(0, 1.01, 0.1)]", "ALPHAS")
open(P, "w").write(src)
print("p7_pcr.py patched")

# Rescore every finished run on GPU 1 (free); p7_partial gets rescored on a rerun once it finishes
sys.path.insert(0, f"{W}/src"); import p7_pcr; importlib.reload(p7_pcr)
for f in sorted(glob.glob(f"{S}/p7_runs/*.json")):
    name = os.path.basename(f)[:-5]
    if json.load(open(f)).get("rescored"): continue
    r = p7_pcr.rescore(name, device="cuda:1"); b = r["dev_best"]
    print(f"{name:<15} Dev F1@7 {b['F1']:.4f} (P {b['P']:.3f} R {b['R']:.3f}) | {b['mode']} α={b['alpha']}"
          f" | dense-only mean {r['dev_dense_only']['mean']['F1']:.4f} maxsim {r['dev_dense_only']['maxsim']['F1']:.4f}")
print("BM25 alone      Dev F1@7 0.3936")

# Back up the fixed code, results and new submissions
subprocess.run(f"nohup python3 {W}/hf_backup.py > {S}/p7_backup_fix.log 2>&1 &", shell=True)

p7_pcr.py patched
p7_partial      Dev F1@7 0.4013 (P 0.437 R 0.371) | mean α=0.55 | dense-only mean 0.3491 maxsim 0.3469
BM25 alone      Dev F1@7 0.3936


CompletedProcess(args='nohup python3 /kaggle/working/hf_backup.py > /kaggle/working/stats/p7_backup_fix.log 2>&1 &', returncode=0)

## Phase 7 — P7-4: QLoRA + mandatory KD student
- QLoRA: frozen M1 with every block Linear quantised to NF4 (Phase-6 NF4Linear) + LoRA r=16 on top.
- KD: teacher = p7_full (12-layer full-FT retriever); student = 6-layer M1 (layers 0,2,4,6,8,10), full FT.
  Loss = 0.5·InfoNCE(gold) + 0.5·T²·KL(teacher ‖ student) over in-batch candidate similarities (T = 2);
  teacher and student see exactly the same sampled chunks.

In [1]:
# ---- P7-4: add QLoRA + KD student to p7_pcr.py, smoke-test both, then launch (GPU 0: KD, GPU 1: QLoRA) ----
import subprocess
W, S = "/kaggle/working", "/kaggle/working/stats"

addon = '''

# ============================================================================= P7-4: QLoRA + knowledge distillation
import torch.nn as nn
import p6_sum as S6                                                     # Phase-6 NF4Linear / LoRA wrapping

LR["qlora"] = 5e-4
_build_base = build


def drop_layers(model, keep):
    """Keep only the given transformer blocks (student = layer subset of M1, initialised from M1)."""
    bb = model.tagger.backbone
    for name, mod in bb.named_modules():
        if isinstance(mod, nn.ModuleList) and len(mod) > max(keep):
            parent = bb.get_submodule(name.rsplit(".", 1)[0]) if "." in name else bb
            setattr(parent, name.rsplit(".", 1)[-1], nn.ModuleList([mod[i] for i in keep]))
            try:
                bb.cfg.n_layers = len(keep)
            except Exception:
                pass
            return name
    raise RuntimeError("no transformer block list found in the backbone")


def build(method, partial_k=4, r=16, alpha=32, student_layers=None):
    """Adds 'qlora' (NF4 frozen base + LoRA) and layer-subset students to the Phase-4 builder."""
    if method == "qlora":
        m = _build_base("full")
        for p in m.parameters():
            p.requires_grad = False
        bb = m.tagger.backbone
        S6.quantize_nf4(bb)
        S6._wrap(bb, lambda l: S6.pp.LoRALinear(l, r, alpha), S6.LORA_TARGETS)
    else:
        m = _build_base(method, partial_k, r, alpha)
    if student_layers:
        drop_layers(m, student_layers)
    return m


def _sample(docs, ids, k):
    """Random k chunks per case (shared by teacher and student so both see identical inputs)."""
    seqs, owner = [], []
    for j, d in enumerate(ids):
        a = docs.arr(d); st = chunk_starts(len(a), 10 ** 9)
        for s in random.sample(st, min(k, len(st))):
            seqs.append(a[s:s + CHUNK]); owner.append(j)
    return seqs, owner


def _embed_seqs(model, seqs, owner, n, device):
    x, mask = cj.collate(seqs, device)
    with cj._autocast(device):
        e = F.normalize(model.embed(x, mask), dim=-1)
    out = torch.zeros(n, e.shape[1], device=device).index_add_(0, torch.tensor(owner, device=device), e)
    return F.normalize(out, dim=-1)


def load_trained(name, method="full", device="cuda"):
    """Rebuild a finished run's model: base M1 + its saved trainable parameters."""
    m = build(method)
    sd = {k: v.float() for k, v in torch.load(f"{CKD}/{name}.pt", map_location="cpu").items()}
    missing = m.load_state_dict(sd, strict=False)
    assert not missing.unexpected_keys, missing.unexpected_keys[:5]
    return m.to(device).eval()


def train_kd(student, teacher, epochs=3, bs=4, k=2, n_hard=2, tau=0.05, lr=1e-4, lam=0.5, T=2.0,
             device="cuda", log=print, seed=0, max_steps=None):
    random.seed(seed); torch.manual_seed(seed)
    q, _ = load_split("Train")
    bm = json.load(open(f"{D7}/bm25_top100_train.json"))
    items = [(qid, r["relevant_candidates"]) for qid, r in q.items() if r["relevant_candidates"]]
    docs = Docs(); params = [p for p in student.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    total = max_steps or epochs * math.ceil(len(items) / bs); warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    scaler = torch.amp.GradScaler("cuda"); student.train(); teacher.eval()
    step, t0, hist, acc = 0, time.time(), [], [0.0, 0.0]
    for ep in range(max(1, epochs)):
        random.shuffle(items)
        for b in range(0, len(items), bs):
            batch = items[b:b + bs]; qids, cand, tgt = [x[0] for x in batch], [], []
            for qid, gold in batch:
                g = set(gold); tgt.append(len(cand)); cand.append(random.choice(gold))
                hard = [c for c in bm[qid][:50] if c not in g and c != qid]
                cand += random.sample(hard, min(n_hard, len(hard)))
            qs, qo = _sample(docs, qids, k); cs, co = _sample(docs, cand, k)
            with torch.no_grad():
                St = _embed_seqs(teacher, qs, qo, len(qids), device) @ _embed_seqs(teacher, cs, co, len(cand), device).T / tau
            Ss = _embed_seqs(student, qs, qo, len(qids), device) @ _embed_seqs(student, cs, co, len(cand), device).T / tau
            bad = torch.zeros_like(Ss, dtype=torch.bool)
            for i, (qid, gold) in enumerate(batch):
                g = set(gold)
                for j, c in enumerate(cand):
                    if j != tgt[i] and (c in g or c == qid):
                        bad[i, j] = True
            Ss, St = Ss.masked_fill(bad, -1e4), St.masked_fill(bad, -1e4)
            ce = F.cross_entropy(Ss, torch.tensor(tgt, device=device))
            kl = F.kl_div(F.log_softmax(Ss / T, -1), F.softmax(St / T, -1), reduction="batchmean") * T * T
            loss = (1 - lam) * ce + lam * kl
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            step += 1; acc[0] += ce.item(); acc[1] += kl.item()
            if step % 50 == 0 or step == total:
                n = min(50, step); hist.append({"step": step, "ce": acc[0] / n, "kl": acc[1] / n}); acc = [0.0, 0.0]
                log(f"   step {step}/{total} | ce {hist[-1]['ce']:.4f} kl {hist[-1]['kl']:.4f} | {time.time()-t0:.0f}s")
            if step >= total:
                return hist
    return hist


def run_kd(name, teacher="p7_full", student_layers=(0, 2, 4, 6, 8, 10), epochs=3, bs=4, k=2, n_hard=2,
           tau=0.05, lr=1e-4, lam=0.5, T=2.0, max_chunks=16, seed=0, log=print, device="cuda"):
    t0 = time.time()
    tm = load_trained(teacher, "full", device)
    st = build("full", student_layers=list(student_layers)).to(device)
    ntr = sum(p.numel() for p in st.parameters() if p.requires_grad); ntot = sum(p.numel() for p in st.parameters())
    log(f"=== {name}: KD {teacher} -> {len(student_layers)}-layer student | {ntot/1e6:.1f}M params | epochs {epochs}")
    torch.cuda.reset_peak_memory_stats()
    hist = train_kd(st, tm, epochs, bs, k, n_hard, tau, lr, lam, T, device, log, seed)
    del tm; torch.cuda.empty_cache(); train_min = (time.time() - t0) / 60
    torch.save({n: p.detach().half().cpu() for n, p in st.named_parameters()}, f"{CKD}/{name}.pt")
    docs, emb, enc_s = Docs(), {}, {}
    for split in ("Dev", "Test"):
        _, cids = load_split(split); t = time.time()
        emb[split] = encode(st, docs, cids, max_chunks, device); enc_s[split] = time.time() - t
        np.savez(f"{EMB}/{name}_{split.lower()}.npz", E=emb[split][0], M=emb[split][1])
        log(f"   encoded {split}: {len(cids)} cases in {enc_s[split]:.0f}s")
    dev, _, _ = score_split(name, *emb["Dev"], "Dev", log, device)
    best = max(dev, key=lambda key: dev[key][0]["F1"])
    test, qids, cids = score_split(name, *emb["Test"], "Test", log, device, fixed=best)
    write_submission(test[best][1], name, qids, cids)
    rec = {"name": name, "method": "kd", "teacher": teacher, "student_layers": list(student_layers),
           "cfg": dict(epochs=epochs, bs=bs, k=k, n_hard=n_hard, tau=tau, lr=lr, lam=lam, T=T, max_chunks=max_chunks),
           "trainable_M": round(ntr / 1e6, 3), "total_M": round(ntot / 1e6, 1),
           "peak_mem_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2), "train_min": round(train_min, 1),
           "encode_s": {s: round(v) for s, v in enc_s.items()},
           "dev_best": {"mode": best[0], "alpha": best[1], **dev[best][0]},
           "dev_dense_only": {m: dev[(m, 1.0)][0] for m in ("mean", "maxsim")},
           "dev_table": {f"{m}|{a}": v[0] for (m, a), v in dev.items()}, "hist": hist,
           "rescored": "fixed fusion", "min": round((time.time() - t0) / 60, 1)}
    json.dump(rec, open(f"{OUT}/{name}.json", "w"), indent=1)
    log(f"DONE {name}: Dev F1 {dev[best][0]['F1']:.4f} | {best[0]} alpha {best[1]} | {rec['min']} min")
    return rec


def smoke2(log=print, device="cuda"):
    docs = Docs(); _, cids = load_split("Dev")
    m = build("qlora").to(device); ntr = sum(p.numel() for p in m.parameters() if p.requires_grad)
    train(m, "qlora", bs=2, k=1, n_hard=1, device=device, log=lambda *a: None, max_steps=2)
    E, M = encode(m, docs, cids[:4], max_chunks=2, device=device)
    log(f"smoke qlora   trainable {ntr/1e6:.3f}M | 2 steps ok | emb {E.shape}"); del m; torch.cuda.empty_cache()
    tm = load_trained("p7_full", "full", device)
    st = build("full", student_layers=[0, 2, 4, 6, 8, 10]).to(device)
    n = sum(p.numel() for p in st.parameters())
    train_kd(st, tm, bs=2, k=1, n_hard=1, device=device, log=lambda *a: None, max_steps=2)
    E, M = encode(st, docs, cids[:4], max_chunks=2, device=device)
    log(f"smoke kd      student {n/1e6:.1f}M params | 2 steps ok | emb {E.shape}")
    log("SMOKE2 PASSED")
'''
open(f"{W}/src/p7_pcr.py", "a").write(addon)

# Runner: KD jobs (they have a "teacher") go to run_kd; add --smoke2
R = open(f"{W}/src/p7_run.py").read()
R = R.replace("P.run(name, log=log, **job)", "(P.run_kd if 'teacher' in job else P.run)(name, log=log, **job)")
R = R.replace('ap.add_argument("--smoke", action="store_true")',
              'ap.add_argument("--smoke", action="store_true"); ap.add_argument("--smoke2", action="store_true")')
R = R.replace("if args.smoke:", "if args.smoke2:\n    P.smoke2(log); sys.exit()\nif args.smoke:")
open(f"{W}/src/p7_run.py", "w").write(R)

# Smoke test (GPU 0), then launch only if it passed
out = subprocess.run(f"cd {W}/src && CUDA_VISIBLE_DEVICES=0 python3 p7_run.py --smoke2 2>&1 | tail -n 20",
                     shell=True, capture_output=True, text=True).stdout
print(out)
if "SMOKE2 PASSED" in out:
    import json
    PLANS = {0: {"name": "p7_kd_m1x6", "teacher": "p7_full", "student_layers": [0, 2, 4, 6, 8, 10], "epochs": 3},
             1: {"name": "p7_qlora", "method": "qlora", "epochs": 3}}
    for gpu, job in PLANS.items():
        pf = f"{S}/p7plan_{job['name']}.json"; json.dump([job], open(pf, "w"))
        chain = (f"cd {W}/src && export CUDA_VISIBLE_DEVICES={gpu}; python3 p7_run.py --plan {pf}; "
                 f"python3 {W}/hf_backup.py {job['name']}")
        subprocess.run(f'nohup bash -c "{chain}" >> {S}/p7_gpu{gpu}.log 2>&1 &', shell=True)
        print(f"launched {job['name']} on GPU {gpu}")
else:
    print("smoke failed - nothing launched; paste the output above")

Traceback (most recent call last):
  File "/kaggle/working/src/p7_run.py", line 19, in <module>
    P.smoke2(log); sys.exit()
    ^^^^^^^^^^^^^
  File "/kaggle/working/src/p7_pcr.py", line 458, in smoke2
    m = build("qlora").to(device); ntr = sum(p.numel() for p in m.parameters() if p.requires_grad)
        ^^^^^^^^^^^^^^
  File "/kaggle/working/src/p7_pcr.py", line 338, in build
    S6.quantize_nf4(bb)
  File "/kaggle/working/src/p6_sum.py", line 207, in quantize_nf4
    for L in layers(m):
             ^^^^^^^^^
  File "/kaggle/working/src/p6_sum.py", line 87, in layers
    return list(m.enc) + list(m.dec) if m.arch == "encdec" else list(m.blocks)
                                        ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1965, in __getattr__
    raise AttributeError(
AttributeError: 'DenseGPT' object has no attribute 'arch'

smoke failed - nothing launched; paste the output above


In [2]:
# ---- P7-4 fix: give the Phase-4 backbone the 'arch' attribute the Phase-6 NF4/LoRA helpers expect, then retry ----
import subprocess, json
W, S = "/kaggle/working", "/kaggle/working/stats"
P = f"{W}/src/p7_pcr.py"; src = open(P).read()

old = "        bb = m.tagger.backbone\n        S6.quantize_nf4(bb)"
new = ("        bb = m.tagger.backbone\n"
       "        if not hasattr(bb, \"arch\"):\n"
       "            bb.arch = m.tagger.arch                                   # p6 helpers read m.arch ('dense' -> m.blocks)\n"
       "        S6.quantize_nf4(bb)")
assert old in src, "qlora block not found - paste the build() code"
open(P, "w").write(src.replace(old, new, 1))
print("patched build('qlora')")

# Smoke test again on GPU 0, launch only if it passes
out = subprocess.run(f"cd {W}/src && CUDA_VISIBLE_DEVICES=0 python3 p7_run.py --smoke2 2>&1 | tail -n 20",
                     shell=True, capture_output=True, text=True).stdout
print(out)
if "SMOKE2 PASSED" in out:
    PLANS = {0: {"name": "p7_kd_m1x6", "teacher": "p7_full", "student_layers": [0, 2, 4, 6, 8, 10], "epochs": 3},
             1: {"name": "p7_qlora", "method": "qlora", "epochs": 3}}
    for gpu, job in PLANS.items():
        pf = f"{S}/p7plan_{job['name']}.json"; json.dump([job], open(pf, "w"))
        chain = (f"cd {W}/src && export CUDA_VISIBLE_DEVICES={gpu}; python3 p7_run.py --plan {pf}; "
                 f"python3 {W}/hf_backup.py {job['name']}")
        subprocess.run(f'nohup bash -c "{chain}" >> {S}/p7_gpu{gpu}.log 2>&1 &', shell=True)
        print(f"launched {job['name']} on GPU {gpu}")
else:
    print("smoke failed - nothing launched; paste the output above")

patched build('qlora')
02:53:24 smoke qlora   trainable 0.786M | 2 steps ok | emb (4, 2, 768)
02:53:31 smoke kd      student 62.3M params | 2 steps ok | emb (4, 2, 768)
02:53:31 SMOKE2 PASSED

launched p7_kd_m1x6 on GPU 0
launched p7_qlora on GPU 1


In [3]:
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader; grep -h -E "===|step|encoded|DONE|FAILED|Error" /kaggle/working/stats/p7_gpu*.log | tail -n 8

0, 0 %, 0 MiB
1, 100 %, 7261 MiB
02:59:47    step 350/615 | loss 1.6743 | 363s
03:00:40    step 400/615 | loss 1.6923 | 416s
03:01:33    step 450/615 | loss 1.5568 | 469s
03:02:26    step 500/615 | loss 1.5092 | 522s
03:03:19    step 550/615 | loss 1.6524 | 576s
03:04:13    step 600/615 | loss 1.5541 | 629s
03:04:28    step 615/615 | loss 0.4512 | 644s
03:06:41    encoded Dev: 1023 cases in 133s


In [5]:
# ---- Back up p7_kd_m1x6 and p7_qlora to HF (their automatic backups ran without the token) ----
import os, subprocess
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_WRITE_TOKEN")      # this kernel only; never printed
out = subprocess.run("python3 /kaggle/working/hf_backup.py p7_kd_m1x6 p7_qlora 2>&1 | grep -E 'backup ok|retry|Error'",
                     shell=True, capture_output=True, text=True).stdout
print(out or "no output - check that hf_backup.py exists")

backup ok ['p7_kd_m1x6', 'p7_qlora']



In [7]:
# ---- Collect all Phase-7 submission zips into one folder, ranked by Dev F1@7 ----
import glob, json, os, shutil
W, S = "/kaggle/working", "/kaggle/working/stats"
up = f"{W}/upload_p7"; os.makedirs(up, exist_ok=True)

runs = sorted(((json.load(open(f))["dev_best"]["F1"], os.path.basename(f)[:-5])
               for f in glob.glob(f"{S}/p7_runs/*.json")), reverse=True)
for k, (f1, n) in enumerate(runs, 1):
    # The writer saves to submissions_p7/p7_<run name>/ - find the zip inside instead of guessing the name
    z = glob.glob(f"{W}/submissions_p7/p7_{n}/*.zip")
    assert len(z) == 1, f"{n}: expected one zip, found {z}"
    shutil.copy(z[0], f"{up}/{k:02d}_{n}.zip")
    print(f"{k:2d}  {n:<15} Dev F1@7 {f1:.4f}")
shutil.copy(f"{W}/submissions_p7/p7_bm25/p7_bm25.zip", f"{up}/99_bm25_baseline.zip")
print("\nzips in upload_p7/ (download from the Output panel)")

 1  p7_lorma_plus   Dev F1@7 0.4091
 2  p7_qlora        Dev F1@7 0.4069
 3  p7_kd_m1x6      Dev F1@7 0.4024
 4  p7_full         Dev F1@7 0.4024
 5  p7_partial      Dev F1@7 0.4013
 6  p7_ia3          Dev F1@7 0.3980
 7  p7_zeroshot     Dev F1@7 0.3969
 8  p7_prompt       Dev F1@7 0.3958

zips in upload_p7/ (download from the Output panel)


## Phase 7 — P7-5: Improvement round
A (GPU, background): LoRMA+ and QLoRA with 3 chunks/case in training (vs 2) and 5 epochs (vs 3).
B (CPU): stronger lexical side tuned on Train — BM25 with numbers + bigrams (citations, sections),
citation-string matching, precedent popularity prior (leave-one-out on Train) — then fused with an
ensemble of every trained model's dense scores (α on Dev). Writes the hybrid test submission.

In [8]:
# ---- P7-5A: launch improved LoRMA+ and QLoRA runs (3 chunks/case in training, 5 epochs) ----
import os, json, subprocess
from kaggle_secrets import UserSecretsClient
W, S = "/kaggle/working", "/kaggle/working/stats"
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_WRITE_TOKEN")      # inherited by the jobs; never printed

PLANS = {0: {"name": "p7_lorma_k3e5", "method": "lorma_plus", "k": 3, "epochs": 5},
         1: {"name": "p7_qlora_k3e5", "method": "qlora",      "k": 3, "epochs": 5}}
for gpu, job in PLANS.items():
    pf = f"{S}/p7plan_{job['name']}.json"; json.dump([job], open(pf, "w"))
    chain = (f"cd {W}/src && export CUDA_VISIBLE_DEVICES={gpu}; python3 p7_run.py --plan {pf}; "
             f"python3 {W}/hf_backup.py {job['name']}")
    subprocess.run(f'nohup bash -c "{chain}" >> {S}/p7_gpu{gpu}.log 2>&1 &', shell=True)
    print(f"launched {job['name']} on GPU {gpu}")

launched p7_lorma_k3e5 on GPU 0
launched p7_qlora_k3e5 on GPU 1


In [9]:
# ---- P7-5B: stronger lexical scoring + citation match + popularity prior + dense ensemble -> hybrid ----
import sys, json, re, glob, time, numpy as np
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer
sys.path.insert(0, "/kaggle/working/src"); import p7_pcr as P

S7 = "/kaggle/working/stats"
# All three splits: queries, ordered pool ids, pool texts (queries are inside their own pool)
SP = {}
for s in ("Train", "Dev", "Test"):
    q, cids = P.load_split(s)
    c = json.load(open(f"{P.R7}/{s}/{s.lower()}_candidates.json"))
    pos = {x: j for j, x in enumerate(cids)}
    SP[s] = dict(q=q, qids=list(q), cids=cids, qrows=np.array([pos[x] for x in q]),
                 text=[" ".join(c[x]["text"]) for x in cids])
    del c

def bm25W(tf, k1, b=1.0):
    """Document-side BM25 weights on a sparse doc x term matrix."""
    tf = tf.tocsr().astype(np.float32, copy=True)
    dl = np.asarray(tf.sum(1)).ravel(); N = tf.shape[0]
    df = np.bincount(tf.indices, minlength=tf.shape[1])
    idf = np.log1p((N - df + 0.5) / (df + 0.5)).astype(np.float32)
    rows = np.repeat(np.arange(N), np.diff(tf.indptr)); v = tf.data
    tf.data = idf[tf.indices] * v * (k1 + 1) / (v + k1 * (1 - b + b * dl[rows] / dl.mean()))
    return tf

def self_nan(S, s):
    """The query's own column -> NaN (excluded from z-scores and from ranking)."""
    S = S.astype(np.float32); S[np.arange(len(SP[s]["qids"])), SP[s]["qrows"]] = np.nan; return S

def f1(S, s):
    d = SP[s]; return P.evaluate(P.top7(np.nan_to_num(S, nan=-np.inf), d["qids"], d["cids"]), d["q"])

# ---------- 1) BM25 variants (queries are rows of the pool matrix, so each split is vectorised once)
VARIANTS = {
    "letters":       lambda: CountVectorizer(token_pattern=r"(?u)\b[a-z][a-z]+\b", stop_words="english"),
    "alnum":         lambda: HashingVectorizer(token_pattern=r"(?u)\b(?:[a-z]{2,}|\d+)\b", stop_words="english",
                                               n_features=2**22, alternate_sign=False, norm=None),
    "alnum+bigrams": lambda: HashingVectorizer(token_pattern=r"(?u)\b(?:[a-z]{2,}|\d+)\b", stop_words="english",
                                               ngram_range=(1, 2), n_features=2**23, alternate_sign=False, norm=None),
}
BM = {}
for vname, mk in VARIANTS.items():
    t = time.time(); X = {}
    for s in SP:
        v = mk(); X[s] = (v.fit_transform if hasattr(v, "fit_transform") and vname == "letters" else v.transform)(SP[s]["text"])
    best = None
    for k1 in (10.0, 20.0, 40.0):                                       # tune k1 on Train
        Q = X["Train"][SP["Train"]["qrows"]].astype(np.float32); Q.data = np.log1p(Q.data)
        St = self_nan((Q @ bm25W(X["Train"], k1).T).toarray(), "Train")
        m = f1(St, "Train")["F1"]
        if best is None or m > best[0]: best = (m, k1)
    k1 = best[1]; BM[vname] = {}
    for s in SP:
        Q = X[s][SP[s]["qrows"]].astype(np.float32); Q.data = np.log1p(Q.data)
        BM[vname][s] = self_nan((Q @ bm25W(X[s], k1).T).toarray(), s)
    print(f"BM25 {vname:<14} k1={k1:<5} Train F1 {best[0]:.4f} | Dev F1 {f1(BM[vname]['Dev'], 'Dev')['F1']:.4f}"
          f"  ({time.time()-t:.0f}s)")
    del X
lex_name = max(BM, key=lambda n: f1(BM[n]["Train"], "Train")["F1"])
print("-> lexical base:", lex_name)

# ---------- 2) Citation matching: candidate's own citation (from its header) found in the query text
CIT = re.compile(r"\(\d{4}\)\s*\d+\s*SCC\s*\d+|\d{4}\s*AIR\s*\d+|AIR\s*\d{4}\s*[A-Z][A-Za-z]*\s*\d+|"
                 r"\d{4}\s*SCR\s*\(\d+\)\s*\d+|\[\d{4}\]\s*\d+\s*SCR\s*\d+")
norm = lambda x: re.sub(r"\s+", "", x).upper()
def own_cites(text):
    m = re.search(r"CITATION:(.*?)(?:CITATOR|ACT:|HEADNOTE|$)", text[:1500])
    return {norm(x) for x in CIT.findall(m.group(1))} if m else set()
CITE = {}
for s in SP:
    d = SP[s]; own = [own_cites(t) for t in d["text"]]
    inq = [{norm(x) for x in CIT.findall(d["text"][r])} - own[r] for r in d["qrows"]]
    M = np.zeros((len(d["qids"]), len(d["cids"])), np.float32)
    for i, cs in enumerate(inq):
        if cs:
            for j, oc in enumerate(own):
                if oc & cs: M[i, j] = 1.0
    CITE[s] = M
    if s != "Test":
        gold = [(i, j) for i, qid in enumerate(d["qids"]) for j, c in enumerate(d["cids"])
                if c in set(d["q"][qid]["relevant_candidates"])]
        hits = sum(M[i, j] for i, j in gold); tot = M.sum()
        print(f"citations {s}: cands with own citation {100*np.mean([bool(o) for o in own]):.0f}% | "
              f"gold pairs matched {100*hits/len(gold):.1f}% | match precision {100*hits/max(tot,1):.1f}% ({int(tot)} matches)")

# ---------- 3) Popularity prior: how many TRAIN queries cite each candidate (leave-one-out on Train)
cnt = Counter(g for r in SP["Train"]["q"].values() for g in r["relevant_candidates"])
PRI = {}
for s in SP:
    d = SP[s]; base = np.array([cnt.get(c, 0) for c in d["cids"]], np.float32)
    M = np.tile(base, (len(d["qids"]), 1))
    if s == "Train":                                                    # remove each query's own labels
        pos = {c: j for j, c in enumerate(d["cids"])}
        for i, qid in enumerate(d["qids"]):
            for g in d["q"][qid]["relevant_candidates"]: M[i, pos[g]] -= 1
    PRI[s] = np.log1p(M)

# ---------- 4) Lexical combination, weights tuned on Train
Z = lambda S: P.znorm(S)
best = (0, 0, 0)
for wp in (0, 0.25, 0.5, 1, 2):
    for wc in (0, 1, 2, 4, 8):
        m = f1(Z(BM[lex_name]["Train"]) + wp * PRI["Train"] + wc * CITE["Train"], "Train")["F1"]
        if m > best[0]: best = (m, wp, wc)
_, wp, wc = best
LEX = {s: Z(BM[lex_name][s]) + wp * PRI[s] + wc * CITE[s] for s in SP}
print(f"lexical combo (Train-tuned): prior w={wp} cite w={wc} | Train {best[0]:.4f} | Dev {f1(LEX['Dev'], 'Dev')}")

# ---------- 5) Dense ensemble from saved embeddings (each run in its own Dev-best pooling), fused with LEX
runs = sorted(((json.load(open(f))["dev_best"]["F1"], json.load(open(f))) for f in glob.glob(f"{S7}/p7_runs/*.json")
               if "zeroshot" not in f), key=lambda x: -x[0])
DZ = {}
for _, r in runs:
    for s in ("Dev", "Test"):
        z = np.load(f"{P.EMB}/{r['name']}_{s.lower()}.npz")
        Sd = P.dense_scores(z["E"], z["M"], list(SP[s]["qrows"]), r["dev_best"]["mode"], "cpu")
        DZ.setdefault(r["name"], {})[s] = Z(self_nan(Sd, s))
results = []
for top in (1, 3, 5, len(runs)):                                       # ensemble of the best `top` models
    names = [r["name"] for _, r in runs[:top]]
    D = {s: np.nanmean([DZ[n][s] for n in names], 0) for s in ("Dev", "Test")}
    for a in P.ALPHAS:
        results.append((f1((1 - a) * LEX["Dev"] + a * Z(D["Dev"]), "Dev")["F1"], top, a))
results.sort(reverse=True)
print("\nbest hybrids on Dev:", *[f"F1 {m:.4f} | top-{t} models | α={a}" for m, t, a in results[:5]], sep="\n  ")

# ---------- 6) Write the hybrid test submission with the best Dev setting
m, top, a = results[0]; names = [r["name"] for _, r in runs[:top]]
Dt = np.nanmean([DZ[n]["Test"] for n in names], 0)
d = SP["Test"]; rank = P.top7(np.nan_to_num((1 - a) * LEX["Test"] + a * Z(Dt), nan=-np.inf), d["qids"], d["cids"])
P.write_submission(rank, "hybrid", d["qids"], d["cids"])
json.dump({"lexical": lex_name, "prior_w": wp, "cite_w": wc, "dense_models": names, "alpha": a,
           "dev": f1((1 - a) * LEX["Dev"] + a * Z(np.nanmean([DZ[n]["Dev"] for n in names], 0)), "Dev")},
          open(f"{S7}/p7_hybrid.json", "w"), indent=1)
print(f"\nwrote submissions_p7/p7_hybrid/  (Dev F1@7 {m:.4f})")

BM25 letters        k1=20.0  Train F1 0.2938 | Dev F1 0.3936  (39s)
BM25 alnum          k1=20.0  Train F1 0.3001 | Dev F1 0.4002  (45s)
BM25 alnum+bigrams  k1=10.0  Train F1 0.3340 | Dev F1 0.4402  (66s)
-> lexical base: alnum+bigrams
citations Train: cands with own citation 32% | gold pairs matched 0.0% | match precision 100.0% (1 matches)
citations Dev: cands with own citation 42% | gold pairs matched 0.0% | match precision 0.0% (0 matches)
lexical combo (Train-tuned): prior w=0 cite w=0 | Train 0.3340 | Dev {'F1': 0.4402445803219046, 'P': 0.4794188861985472, 'R': 0.40698869475847893}


/tmp/ipykernel_58/847123294.py:123: RuntimeWarning: Mean of empty slice
  D = {s: np.nanmean([DZ[n][s] for n in names], 0) for s in ("Dev", "Test")}



best hybrids on Dev:
  F1 0.4425 | top-1 models | α=0.15
  F1 0.4414 | top-7 models | α=0.05
  F1 0.4414 | top-5 models | α=0.05
  F1 0.4414 | top-3 models | α=0.1
  F1 0.4414 | top-3 models | α=0.05

wrote submissions_p7/p7_hybrid/  (Dev F1@7 0.4425)


/tmp/ipykernel_58/847123294.py:131: RuntimeWarning: Mean of empty slice
  Dt = np.nanmean([DZ[n]["Test"] for n in names], 0)
/tmp/ipykernel_58/847123294.py:135: RuntimeWarning: Mean of empty slice
  "dev": f1((1 - a) * LEX["Dev"] + a * Z(np.nanmean([DZ[n]["Dev"] for n in names], 0)), "Dev")},


## Phase 7 — P7-5C: Tune the n-gram BM25, pseudo-relevance feedback, re-fuse every technique
Tuned on Train: n-gram order (1–2 vs 1–3), k1, b; then neighbour smoothing
score'(q,c) = z(S)(q,c) + β · mean over the query's top-k candidates d of z(DocDoc)(d,c).
Every trained model is then re-fused with this lexical score (α on Dev) → p7_<run>_v2 submissions + hybrid_v2.

In [10]:
# ---- P7-5C: tune the n-gram BM25 + neighbour PRF on Train; re-fuse all techniques with it ----
import warnings, json, glob, time, numpy as np
from sklearn.feature_extraction.text import HashingVectorizer
warnings.filterwarnings("ignore", message="Mean of empty slice")       # the query's own (NaN) column
S7 = "/kaggle/working/stats"; Z = P.znorm

def vectorise(ngram):
    v = HashingVectorizer(token_pattern=r"(?u)\b(?:[a-z]{2,}|\d+)\b", stop_words="english",
                          ngram_range=(1, ngram), n_features=2**23, alternate_sign=False, norm=None)
    return {s: v.transform(SP[s]["text"]).astype(np.float32).tocsr() for s in SP}

def qmat(X, rows):
    Q = X[rows].copy(); Q.data = np.log1p(Q.data); return Q

# ---------- 1) n-gram order, k1, b on Train
grid = []
for ng in (2, 3):
    t = time.time(); X = vectorise(ng)
    for k1 in (5.0, 7.0, 10.0):
        for b in (0.9, 1.0):
            St = self_nan((qmat(X["Train"], SP["Train"]["qrows"]) @ bm25W(X["Train"], k1, b).T).toarray(), "Train")
            grid.append((f1(St, "Train")["F1"], ng, k1, b))
    print(f"ngram 1-{ng} done in {time.time()-t:.0f}s"); del X
grid.sort(reverse=True)
print("Train top 4:", *[f"F1 {m:.4f} | 1-{n}gram k1={k} b={b}" for m, n, k, b in grid[:4]], sep="\n  ")
_, ng, k1, b = grid[0]

# Final lexical scores per split + the pool x pool similarity used for feedback
X = vectorise(ng); BASE, DD = {}, {}
for s in SP:
    W = bm25W(X[s], k1, b)
    BASE[s] = Z(self_nan((qmat(X[s], SP[s]["qrows"]) @ W.T).toarray(), s))
    dd = (qmat(X[s], np.arange(X[s].shape[0])) @ W.T).toarray().astype(np.float32)
    np.fill_diagonal(dd, np.nan); DD[s] = Z(dd)
del X
print(f"lexical 1-{ng}gram k1={k1} b={b}: Dev {f1(BASE['Dev'], 'Dev')['F1']:.4f}")

# ---------- 2) Neighbour smoothing (pseudo-relevance feedback), tuned on Train
def prf(s, beta, k):
    B = BASE[s]
    if beta == 0: return B
    top = np.argsort(-np.nan_to_num(B, nan=-np.inf), axis=1)[:, :k]           # each query's top-k candidates
    return B + beta * np.nanmean(DD[s][top], axis=1)                            # (nq, k, n) -> (nq, n)
best = max(((f1(prf("Train", be, k), "Train")["F1"], be, k) for be in (0, 0.1, 0.2, 0.3, 0.5, 0.8) for k in (3, 5, 10)))
_, beta, kprf = best
LEX2 = {s: prf(s, beta, kprf) for s in SP}
print(f"PRF beta={beta} k={kprf} (Train {best[0]:.4f}) -> Dev {f1(LEX2['Dev'], 'Dev')}")

# ---------- 3) Re-fuse every trained technique with the new lexical score (alpha on Dev); write *_v2 submissions
def dense_z(name, mode):
    if name not in DZ:                                                          # runs finished after P7-5B
        DZ[name] = {}
        for s in ("Dev", "Test"):
            z = np.load(f"{P.EMB}/{name}_{s.lower()}.npz")
            DZ[name][s] = Z(self_nan(P.dense_scores(z["E"], z["M"], list(SP[s]["qrows"]), mode, "cpu"), s))
    return DZ[name]

recs = {}
for f in sorted(glob.glob(f"{S7}/p7_runs/*.json")):
    r = json.load(open(f)); n = r["name"]; D = dense_z(n, r["dev_best"]["mode"])
    m, a = max((f1((1 - a) * LEX2["Dev"] + a * D["Dev"], "Dev")["F1"], a) for a in P.ALPHAS)
    d = SP["Test"]
    P.write_submission(P.top7(np.nan_to_num((1 - a) * LEX2["Test"] + a * D["Test"], nan=-np.inf), d["qids"], d["cids"]),
                       f"{n}_v2", d["qids"], d["cids"])
    recs[n] = {"dev_F1_v2": m, "alpha": a, "dev_F1_v1": r["dev_best"]["F1"]}
    print(f"{n:<16} v1 {r['dev_best']['F1']:.4f} -> v2 {m:.4f}  (α={a})")

# ---------- 4) Hybrid v2: ensemble of the best models' dense scores + new lexical
order = sorted(recs, key=lambda n: -recs[n]["dev_F1_v2"]); res = []
for top in (1, 3, 5):
    Dd = np.nanmean([DZ[n]["Dev"] for n in order[:top]], 0)
    res += [(f1((1 - a) * LEX2["Dev"] + a * Z(Dd), "Dev")["F1"], top, a) for a in P.ALPHAS]
m, top, a = max(res)
Dt = np.nanmean([DZ[n]["Test"] for n in order[:top]], 0); d = SP["Test"]
P.write_submission(P.top7(np.nan_to_num((1 - a) * LEX2["Test"] + a * Z(Dt), nan=-np.inf), d["qids"], d["cids"]),
                   "hybrid_v2", d["qids"], d["cids"])
json.dump({"lexical": {"ngram": ng, "k1": k1, "b": b, "prf_beta": beta, "prf_k": kprf},
           "per_run": recs, "hybrid": {"models": order[:top], "alpha": a, "dev_F1": m}},
          open(f"{S7}/p7_v2.json", "w"), indent=1)
print(f"\nhybrid_v2: top-{top} models α={a} -> Dev F1@7 {m:.4f}  (written to submissions_p7/p7_hybrid_v2/)")

ngram 1-2 done in 72s
ngram 1-3 done in 112s
Train top 4:
  F1 0.3437 | 1-3gram k1=10.0 b=1.0
  F1 0.3394 | 1-3gram k1=7.0 b=1.0
  F1 0.3381 | 1-3gram k1=5.0 b=1.0
  F1 0.3340 | 1-2gram k1=10.0 b=1.0
lexical 1-3gram k1=10.0 b=1.0: Dev 0.4447
PRF beta=0 k=10 (Train 0.3437) -> Dev {'F1': 0.4446914952746562, 'P': 0.48426150121065376, 'R': 0.41109969167523125}
p7_full          v1 0.4024 -> v2 0.4480  (α=0.25)
p7_ia3           v1 0.3980 -> v2 0.4458  (α=0.5)
p7_kd_m1x6       v1 0.4024 -> v2 0.4480  (α=0.65)
p7_lorma_k3e5    v1 0.4113 -> v2 0.4469  (α=0.35)
p7_lorma_plus    v1 0.4091 -> v2 0.4458  (α=0.3)
p7_partial       v1 0.4013 -> v2 0.4458  (α=0.25)
p7_prompt        v1 0.3958 -> v2 0.4447  (α=0.05)
p7_qlora         v1 0.4069 -> v2 0.4480  (α=0.35)
p7_zeroshot      v1 0.3969 -> v2 0.4469  (α=0.25)

hybrid_v2: top-3 models α=0.65 -> Dev F1@7 0.4480  (written to submissions_p7/p7_hybrid_v2/)


In [11]:
# ---- Collect the v2 submissions (all techniques re-fused with the trigram lexical score) + hybrid_v2 ----
import glob, json, os, shutil
W, S = "/kaggle/working", "/kaggle/working/stats"
up = f"{W}/upload_p7_v2"; os.makedirs(up, exist_ok=True)
v2 = json.load(open(f"{S}/p7_v2.json"))
shutil.copy(glob.glob(f"{W}/submissions_p7/p7_hybrid_v2/*.zip")[0], f"{up}/00_hybrid_v2.zip")
for k, (n, r) in enumerate(sorted(v2["per_run"].items(), key=lambda x: -x[1]["dev_F1_v2"]), 1):
    shutil.copy(glob.glob(f"{W}/submissions_p7/p7_{n}_v2/*.zip")[0], f"{up}/{k:02d}_{n}_v2.zip")
    print(f"{k:2d}  {n:<16} Dev F1@7 {r['dev_F1_v2']:.4f}")
print("hybrid_v2 Dev F1@7", round(v2["hybrid"]["dev_F1"], 4), "\nzips in upload_p7_v2/")
subprocess.run(f"nohup python3 {W}/hf_backup.py > {S}/p7_backup_v2.log 2>&1 &", shell=True) if (subprocess := __import__("subprocess")) else None

 1  p7_full          Dev F1@7 0.4480
 2  p7_kd_m1x6       Dev F1@7 0.4480
 3  p7_qlora         Dev F1@7 0.4480
 4  p7_lorma_k3e5    Dev F1@7 0.4469
 5  p7_zeroshot      Dev F1@7 0.4469
 6  p7_ia3           Dev F1@7 0.4458
 7  p7_lorma_plus    Dev F1@7 0.4458
 8  p7_partial       Dev F1@7 0.4458
 9  p7_prompt        Dev F1@7 0.4447
hybrid_v2 Dev F1@7 0.448 
zips in upload_p7_v2/


CompletedProcess(args='nohup python3 /kaggle/working/hf_backup.py > /kaggle/working/stats/p7_backup_v2.log 2>&1 &', returncode=0)

In [12]:
   !python3 /kaggle/working/hf_backup.py 2>&1 | grep -E "backup ok|retry"

backup ok []


In [13]:
# ---- v2 for p7_qlora_k3e5: re-fuse with the trigram lexical score, refresh hybrid_v2, back up ----
import json, glob, subprocess, numpy as np
W, S7 = "/kaggle/working", "/kaggle/working/stats"
n = "p7_qlora_k3e5"
r = json.load(open(f"{S7}/p7_runs/{n}.json"))                       # fails here if the run didn't finish
print(f"{n}: v1 Dev F1@7 {r['dev_best']['F1']:.4f} | dense-only mean {r['dev_dense_only']['mean']['F1']:.4f} "
      f"maxsim {r['dev_dense_only']['maxsim']['F1']:.4f} | train {r['train_min']} min")

# Re-fuse with the new lexical score, alpha on Dev, write its v2 test submission
D = dense_z(n, r["dev_best"]["mode"])
m, a = max((f1((1 - a) * LEX2["Dev"] + a * D["Dev"], "Dev")["F1"], a) for a in P.ALPHAS)
d = SP["Test"]
P.write_submission(P.top7(np.nan_to_num((1 - a) * LEX2["Test"] + a * D["Test"], nan=-np.inf), d["qids"], d["cids"]),
                   f"{n}_v2", d["qids"], d["cids"])
recs[n] = {"dev_F1_v2": m, "alpha": a, "dev_F1_v1": r["dev_best"]["F1"]}
print(f"{n}: v2 Dev F1@7 {m:.4f} (α={a})")

# Rebuild hybrid_v2 with this model included; keep whichever ensemble is best on Dev
order = sorted(recs, key=lambda x: -recs[x]["dev_F1_v2"]); res = []
for top in (1, 3, 5):
    Dd = np.nanmean([DZ[x]["Dev"] for x in order[:top]], 0)
    res += [(f1((1 - a) * LEX2["Dev"] + a * Z(Dd), "Dev")["F1"], top, a) for a in P.ALPHAS]
mh, top, ah = max(res)
Dt = np.nanmean([DZ[x]["Test"] for x in order[:top]], 0)
P.write_submission(P.top7(np.nan_to_num((1 - ah) * LEX2["Test"] + ah * Z(Dt), nan=-np.inf), d["qids"], d["cids"]),
                   "hybrid_v2", d["qids"], d["cids"])
v2 = json.load(open(f"{S7}/p7_v2.json"))
v2["per_run"] = recs; v2["hybrid"] = {"models": order[:top], "alpha": ah, "dev_F1": mh}
json.dump(v2, open(f"{S7}/p7_v2.json", "w"), indent=1)
print(f"hybrid_v2: top-{top} {order[:top]} α={ah} -> Dev F1@7 {mh:.4f}")

# Back up everything (v2 submissions, p7_v2.json, this run)
print(subprocess.run(f"python3 {W}/hf_backup.py {n} 2>&1 | grep -E 'backup ok|retry'",
                     shell=True, capture_output=True, text=True).stdout)

p7_qlora_k3e5: v1 Dev F1@7 0.4113 | dense-only mean 0.3580 maxsim 0.3580 | train 26.6 min
p7_qlora_k3e5: v2 Dev F1@7 0.4480 (α=0.35)
hybrid_v2: top-3 ['p7_full', 'p7_kd_m1x6', 'p7_qlora'] α=0.65 -> Dev F1@7 0.4480
backup ok ['p7_qlora_k3e5']



In [ ]:
# ---- Collect the v2 submissions (every technique re-fused with the trigram lexical score) + hybrid_v2 ----
import glob, json, os, shutil, subprocess
W, S = "/kaggle/working", "/kaggle/working/stats"
up = f"{W}/upload_p7_v2"
shutil.rmtree(up, ignore_errors=True); os.makedirs(up)                 # start clean: no stale zips

v2 = json.load(open(f"{S}/p7_v2.json"))
shutil.copy(glob.glob(f"{W}/submissions_p7/p7_hybrid_v2/*.zip")[0], f"{up}/00_hybrid_v2.zip")
print(f" 0  hybrid_v2        Dev F1@7 {v2['hybrid']['dev_F1']:.4f}  ({', '.join(v2['hybrid']['models'])})")

for k, (n, r) in enumerate(sorted(v2["per_run"].items(), key=lambda x: -x[1]["dev_F1_v2"]), 1):
    z = glob.glob(f"{W}/submissions_p7/p7_{n}_v2/*.zip")
    assert len(z) == 1, f"{n}: expected one v2 zip, found {z}"
    shutil.copy(z[0], f"{up}/{k:02d}_{n}_v2.zip")
    print(f"{k:2d}  {n:<16} Dev F1@7 {r['dev_F1_v2']:.4f}")

print(f"\n{len(os.listdir(up))} zips in upload_p7_v2/ - download them from the Output panel")
print(subprocess.run(f"python3 {W}/hf_backup.py 2>&1 | grep -E 'backup ok|retry'",
                     shell=True, capture_output=True, text=True).stdout)

 0  hybrid_v2        Dev F1@7 0.4480  (p7_full, p7_kd_m1x6, p7_qlora)
 1  p7_full          Dev F1@7 0.4480
 2  p7_kd_m1x6       Dev F1@7 0.4480
 3  p7_qlora         Dev F1@7 0.4480
 4  p7_qlora_k3e5    Dev F1@7 0.4480
 5  p7_lorma_k3e5    Dev F1@7 0.4469
 6  p7_zeroshot      Dev F1@7 0.4469
 7  p7_ia3           Dev F1@7 0.4458
 8  p7_lorma_plus    Dev F1@7 0.4458
 9  p7_partial       Dev F1@7 0.4458
10  p7_prompt        Dev F1@7 0.4447

11 zips in upload_p7_v2/ - download them from the Output panel
